# 第 63 课｜PPO 的 minibatch、梯度累积与 KL 诊断

**从“整份 rollout 更新一次”，走向“逐步打乱、处理短尾和真正等价的 microbatch 累积”。**

上一课把 Actor、Critic、GAE、PPO-Clip、值函数裁剪连成一条训练链。本课不重复发明新的 PPO 目标，重点研究**工程语义会不会偷偷改变算法**：

1. 不同长度 episode 摊平后，样本、损失与日志的分母如何定义？
2. 每次 minibatch 是否必须覆盖所有 transition？如何处理最后不满一批？
3. 把一批拆成 5+5+5+2 个样本时，怎样累积才与完整17样本更新等价？
4. 采样近似 KL、访问状态上的精确 KL、clip fraction 分别能说明什么？
5. 何时停止 Actor，Critic 是否需要一起停？

本 Notebook 内嵌**独立完整代码**，不依赖第 62 课或配套 `.py`。Python 3.10+、PyTorch，默认 CPU float64，离线运行，不需要下载模型或联网。

**范围：**3个非终止状态、2种动作的合成连续 MDP，回合受外部时间上限截断。真实执行独立 Actor/Critic 的 PPO-Clip、GAE 与一个明确的 `clipmax` critic 变体；不是预训练大模型、分布式 PPO 或 GPU benchmark。

## 0. 建议分两次学习

第一次学到第 8 节，完成数学反例与一次真实 batch 的梯度比对；第二次完成 minibatch训练、KL诊断、练习和独立检查。

**推荐执行方式：**选择已有 PyTorch 环境的 Python 3 内核，顺序运行所有代码单元。代码较长的实现单元默认可折叠；只读短实验输出并不能代替理解对应检查函数。练习开关默认关闭，因此“运行全部单元”不等于完成练习。

完整脚本入口：

```bash
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --checks-only
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --math-only
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py
```

**本课不会下载模型，不会把大模型结果冒充小模型训练结果。** 所有默认数值在代码中生成并保存在新建的时间戳目录。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第63课：PPO minibatch、梯度累积与KL诊断。

Python 3.10+ / PyTorch；离线 CPU 的3状态2动作教学 Actor-Critic。
同一rollout先冻结全局GAE/return/old logp/old value，再分epoch打乱步级minibatch。
精确加权的microbatch累积与普通minibatch在一次更新上等价；
检查精确old-state KL、负的sampled KL、短尾batch与Actor停止的含义。
非预训练LLM、不含并行采样或完整分布式PPO；不证明真实算法优劣。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需已有PyTorch环境；请使用匹配的Jupyter内核") from exc

VERSION = "lesson63-minibatch-accum-kl-v1"
SOURCES = {
    "ppo_paper": "https://arxiv.org/abs/1707.06347",
    "spinningup_ppo": "https://spinningup.openai.com/en/latest/algorithms/ppo.html",
    "spinningup_code": "https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html",
    "sb3_code": "https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html",
    "gae": "https://arxiv.org/abs/1506.02438",
    "gymnasium_time_limit": "https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/",
    "pytorch_autograd": "https://docs.pytorch.org/docs/stable/notes/autograd.html",
}

class LessonError(ValueError):
    """本课程协议错误；不静默修改任务定义。"""

def require(condition, message):
    if not bool(condition):
        raise LessonError(message)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def sha256(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def tensor_hash(named):
    h = hashlib.sha256()
    for name, tensor in sorted(named.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()

def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)

@dataclass(frozen=True)
class Config:
    seed: int = 6301
    gamma: float = .95
    gae_lambda: float = .95
    horizon: int = 6
    rollouts: int = 24
    episodes_per_rollout: int = 25
    epochs: int = 3
    minibatch_size: int = 17
    microbatch_size: int = 5
    actor_lr: float = .012
    critic_lr: float = .010
    actor_hidden: int = 24
    critic_hidden: int = 24
    clip_ratio: float = .2
    value_clip: float = .2
    entropy_coef: float = .006
    target_kl: float = .06
    grad_clip: float = 1.
    eval_episodes: int = 120

    def validate(self):
        for name in ('seed','horizon','rollouts','episodes_per_rollout','epochs',
                     'minibatch_size','microbatch_size','actor_hidden','critic_hidden','eval_episodes'):
            v=getattr(self,name)
            require(type(v) is int and v >= (0 if name=='seed' else 1),name+'必须为整数')
        require(3<=self.horizon<=16 and 1<=self.epochs<=8, 'horizon/epochs超出教学范围')
        require(2<=self.minibatch_size<=512 and 1<=self.microbatch_size<=self.minibatch_size,
                'microbatch必须小于等于minibatch')
        for name in ('gamma','gae_lambda','actor_lr','critic_lr','clip_ratio','value_clip',
                     'entropy_coef','target_kl','grad_clip'):
            v=getattr(self,name)
            require(type(v) in (float,int) and math.isfinite(v),name+'必须有限数值')
        require(0<self.gamma<1 and 0<=self.gae_lambda<=1,'gamma或lambda非法')
        require(self.actor_lr>0 and self.critic_lr>0 and 0<self.clip_ratio<1
                and self.value_clip>0 and self.entropy_coef>=0 and self.target_kl>0
                and self.grad_clip>0,'训练/裁剪参数非法')

In [2]:
torch.set_num_threads(1)
seed_all(6301)
OUTPUT_ROOT = Path("lesson_63_outputs")
RUN_TRAINING = True
cfg = Config()
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("device: cpu / dtype: float64")
print("配置:", json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

Python: 3.13.5 PyTorch: 2.10.0+cpu
device: cpu / dtype: float64
配置: {
  "seed": 6301,
  "gamma": 0.95,
  "gae_lambda": 0.95,
  "horizon": 6,
  "rollouts": 24,
  "episodes_per_rollout": 25,
  "epochs": 3,
  "minibatch_size": 17,
  "microbatch_size": 5,
  "actor_lr": 0.012,
  "critic_lr": 0.01,
  "actor_hidden": 24,
  "critic_hidden": 24,
  "clip_ratio": 0.2,
  "value_clip": 0.2,
  "entropy_coef": 0.006,
  "target_kl": 0.06,
  "grad_clip": 1.0,
  "eval_episodes": 120
}


## 1. 先区分“一个 episode”与“一个 transition”

一个 rollout 可以包含多个不同长度的回合。成功到达目标的回合会真实终止；到了外部时间上限仍未成功的回合是截断。

上一课建立的 GAE 在截断点保留 `V(next_state)` 作 bootstrap，但不会把后面新回合的优势接起来。**本课保留这一规则，绝不把连续的 rollout 缓冲数组误看成一条巨大轨迹。** [Gymnasium 时间限制](https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/)

另一方面，不同统计量的单位不一样：

| 指标 | 自然分母 |
|---|---|
| 六步内成功率 | 实际 episode 数 |
| 每transition的policy loss、entropy、clip fraction | rollout中实际 transition 数 |
| 优化器更新数 | 真正执行了 `optimizer.step()` 的次数 |
| 一批数据重复训练三epoch | 重复观察 `3×steps` 个训练位置，**不是3倍新的采样数据** |

如果一个 episode 长1步、另一个长5步，前者每步指标为1、后者为0，则按回合等权结果为0.5，按transition等权只有1/6。两者都能计算，但在回答不同问题。

In [3]:
GOAL=3
ACTION_WAIT, ACTION_ADVANCE=0,1
STEP_COST,SUCCESS_BONUS=-.01,1.0


@dataclass(frozen=True)
class Step:
    obs: int
    action: int
    reward: float
    next_obs: int
    terminated: bool
    truncated: bool
    old_logp: float

def validate_step(step):
    require(isinstance(step, Step), "Step类型错误")
    require(type(step.obs) is int and 0 <= step.obs < GOAL, "当前obs必须是非终止位置")
    require(type(step.next_obs) is int and 0 <= step.next_obs <= GOAL, "next_obs无效")
    require(type(step.action) is int and step.action in (ACTION_WAIT, ACTION_ADVANCE), "动作非法")
    require(type(step.terminated) is bool and type(step.truncated) is bool,
            "terminated/truncated必须是bool")
    require(not (step.terminated and step.truncated), "真实终止与外部截断在本课程中互斥")
    require(step.next_obs == min(step.obs + step.action, GOAL), "环境动态不匹配")
    require(step.terminated == (step.next_obs == GOAL), "终止状态不匹配")
    expected_reward = STEP_COST + (SUCCESS_BONUS if step.terminated else 0)
    require(math.isfinite(step.reward) and abs(step.reward - expected_reward) < 1e-10,
            "奖励必须来自固定环境，而不是模型自报")
    require(type(step.old_logp) is float and math.isfinite(step.old_logp) and step.old_logp <= 1e-6,
            "需要有效的行为log概率")

def validate_episode(episode, horizon):
    require(type(episode) is list and 1 <= len(episode) <= horizon, "episode长度错误")
    require(episode[0].obs == 0, "每回合必须从位置0开始")
    for i, step in enumerate(episode):
        validate_step(step)
        if i < len(episode) - 1:
            require(not step.terminated and not step.truncated, "回合结束后仍有步骤")
            require(episode[i + 1].obs == step.next_obs, "下一步观察不连续")
    require(episode[-1].terminated or episode[-1].truncated, "回合尾部必须有明确终止状态")
    require(not episode[-1].truncated or len(episode) == horizon,
            "外部时间上限只能在约定horizon触发")

def transition(pos, action, t, horizon):
    require(type(pos) is int and 0 <= pos < GOAL, "pos非法")
    require(type(action) is int and action in (0, 1), "action非法")
    require(type(t) is int and 0 <= t < horizon, "时间步非法")
    nxt = min(pos + action, GOAL)
    terminated = nxt == GOAL
    truncated = t + 1 >= horizon and not terminated
    reward = STEP_COST + (SUCCESS_BONUS if terminated else 0.)
    return nxt, reward, terminated, truncated

def checked_numeric(values, name):
    require(isinstance(values, (list, tuple)) and len(values) > 0, name + "必须为非空序列")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), name + "必须有限")

def discounted_returns(rewards, gamma, bootstrap=0.0):
    """时间倒序累计，最后一个非终止cut使用V(final_obs)作为尾部。"""
    checked_numeric(rewards, "rewards")
    require(type(gamma) in (int, float) and 0 < gamma <= 1, "gamma非法")
    require(type(bootstrap) in (int, float) and math.isfinite(bootstrap), "bootstrap非法")
    acc = float(bootstrap)
    output = [0.] * len(rewards)
    for i in range(len(rewards)-1, -1, -1):
        acc = float(rewards[i]) + gamma*acc
        output[i] = acc
    return output

def generalized_advantage_estimation(episodes, values, next_values, gamma, lam):
    """独立episode：delta=r+gamma*(1-terminated)V(next)-V(s)。

    截断时delta保留V(final_obs)，但由于下一条记录是新回合，trace递推停止。
    自然rollout数组末尾也停止trace，若未终止则必须保留下一状态bootstrap。
    返回 A 与 lambda-return=A+V_old，不连接critic梯度。
    """
    require(type(episodes) is list and bool(episodes), "需要回合列表")
    require(type(gamma) in (float, int) and 0 < gamma <= 1, "gamma非法")
    require(type(lam) in (float, int) and 0 <= lam <= 1, "lambda非法")
    count = sum(len(ep) for ep in episodes)
    checked_numeric(values, "values")
    checked_numeric(next_values, "next_values")
    require(len(values) == len(next_values) == count, "值函数数组长度与步骤数量不符")
    advantages, targets, deltas = [], [], []
    offset = 0
    for ep in episodes:
        require(len(ep) > 0, "空回合")
        for j, step in enumerate(ep):
            validate_step(step)
            if j < len(ep)-1:
                require(not (step.terminated or step.truncated), "episode结束后仍有下一条步骤")
                require(ep[j+1].obs == step.next_obs, "GAE回合内部不连续")
        local_values = values[offset:offset + len(ep)]
        local_next = next_values[offset:offset + len(ep)]
        offset += len(ep)
        local_d = [step.reward + gamma*(0. if step.terminated else nv) - v
                   for step,v,nv in zip(ep,local_values,local_next)]
        local_adv = [0.] * len(ep)
        ahead = 0.
        for j in range(len(ep)-1, -1, -1):
            # 回合结束或采样切断后，不能把下一回合的优势连接过来。
            boundary = ep[j].terminated or ep[j].truncated
            ahead = local_d[j] + gamma*lam*(0. if boundary else ahead)
            local_adv[j] = ahead
        deltas.extend(local_d)
        advantages.extend(local_adv)
        targets.extend([a+v for a,v in zip(local_adv,local_values)])
    return {"delta":deltas, "advantage":advantages, "lambda_target":targets}

def exact_policy_values(forward_probabilities, gamma=0.95):
    """Bellman: 在底层无限时域、位置马尔可夫过程里计算精确V^pi。

    外部6步截断不属于底层MDP，所以不能把它写进该值函数。
    """
    checked_numeric(forward_probabilities, "forward_probabilities")
    require(len(forward_probabilities) == GOAL, "每个非终止位置一个前进概率")
    require(all(0 <= v <= 1 for v in forward_probabilities), "动作概率不在0..1")
    require(type(gamma) in (int,float) and 0 < gamma < 1, "无穷时域要求gamma小于1")
    values = [0.] * GOAL
    for s in range(GOAL-1, -1, -1):
        p = float(forward_probabilities[s])
        forward_value = SUCCESS_BONUS + STEP_COST if s == GOAL-1 else STEP_COST + gamma*values[s+1]
        values[s] = (p*forward_value + (1-p)*STEP_COST)/(1 - gamma*(1-p))
    return values

def finite_horizon_success_probability(forward_probabilities, horizon):
    """固定外部horizon内到达目标的精确概率，不等于底层无限时域V。"""
    require(type(horizon) is int and horizon >= 1, "horizon非法")
    checked_numeric(forward_probabilities, "policy")
    require(len(forward_probabilities)==GOAL and all(0 <= x <= 1 for x in forward_probabilities),
            "策略概率非法")
    active = [1., 0., 0.]
    finished = 0.
    for _ in range(horizon):
        new = [0.] * GOAL
        for s,pop in enumerate(active):
            prob = forward_probabilities[s]
            new[s] += pop*(1-prob)
            if s == GOAL-1:
                finished += pop*prob
            else:
                new[s+1] += pop*prob
        active = new
    require(abs(finished+sum(active)-1)<1e-9, "总概率不归一")
    return finished

In [4]:
toy_ep = [
    [Step(0,1,-.01,1,False,False,math.log(.5)),
     Step(1,0,-.01,1,False,True,math.log(.5))],
]
toy = generalized_advantage_estimation(toy_ep, [.2,.3], [.3,.4], .95,.95)
print("外部截断时仍保留bootstrap的delta:", toy["delta"])
print("trace在末尾停止的GAE:", toy["advantage"])
print("最后一步：",toy["advantage"][-1],"应等于delta:",toy["delta"][-1])

外部截断时仍保留bootstrap的delta: [0.07499999999999996, 0.07]
trace在末尾停止的GAE: [0.13817499999999996, 0.07]
最后一步： 0.07 应等于delta: 0.07


## 2. 独立 Actor/Critic 与按当前策略采样

模型仍用第62课的小型环境：`WAIT=0`、`ADVANCE=1`，目标位置是3。Actor 输出两个动作的 logits，Critic 输出一个状态价值。

采样时保存每一步的 `old_logp`、奖励、`terminated` 和 `truncated`。冻结旧 Critic 的值后统一计算 GAE 与 λ-return。Critic 在本轮更新后，**不能重新计算 GAE 来替换刚才的训练目标**。

本课程的优势归一化发生在**整个 rollout 摊平以后、所有 epoch 开始之前**：


`normalized_advantage = (raw_advantage - mean) / (population_std + epsilon)`

这与某些库的“按 minibatch 重新归一化”约定不同，不能在比较时假定二者完全等价。本课明确选择全 rollout 一次性归一，便于检查 microbatch 与 minibatch 的严格等价。

In [5]:
class Actor(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,2))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "actor状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "actor状态越界")
        return self.net(states)

class Critic(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,1))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "critic状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "critic状态越界")
        return self.net(states).squeeze(-1)

def actor_probabilities(actor):
    with torch.no_grad():
        actor.eval()
        probs=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)[:,ACTION_ADVANCE]
    return probs.tolist()

def value_prediction(critic):
    with torch.no_grad():
        critic.eval()
        return critic(torch.arange(GOAL,dtype=torch.long)).tolist()

def value_error(critic,actor,gamma):
    reference=exact_policy_values(actor_probabilities(actor),gamma)
    prediction=value_prediction(critic)
    return {"predicted":prediction,"exact_for_current_policy":reference,
            "rmse":math.sqrt(sum((a-b)**2 for a,b in zip(prediction,reference))/GOAL),
            "scope":"exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return"}

def collect_rollouts(actor, episodes, horizon, rng):
    require(type(episodes) is int and episodes >= 1, "episodes非法")
    require(type(horizon) is int and horizon >= 3, "horizon非法")
    result=[]
    actor.eval()
    with torch.no_grad():
        p=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)
    for _ in range(episodes):
        pos=0;ep=[]
        for t in range(horizon):
            action=1 if rng.random()<float(p[pos,1]) else 0
            logp=math.log(float(p[pos,action]))
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,logp))
            if terminated or truncated:
                break
            pos=nxt
        validate_episode(ep,horizon)
        result.append(ep)
    return result

def compute_targets(episodes,critic,gamma,lam,bootstrap_truncation=True):
    require(type(bootstrap_truncation) is bool, "bootstrap标志必须为bool")
    states=[s.obs for ep in episodes for s in ep]
    nextstates=[s.next_obs if not s.terminated else 0 for ep in episodes for s in ep]
    with torch.no_grad():
        values=critic(torch.tensor(states,dtype=torch.long)).tolist()
        nextvalues=critic(torch.tensor(nextstates,dtype=torch.long)).tolist()
    for i,step in enumerate(s for ep in episodes for s in ep):
        if step.terminated or (step.truncated and not bootstrap_truncation):
            nextvalues[i]=0.
    target=generalized_advantage_estimation(episodes,values,nextvalues,gamma,lam)
    return {"states":states,"next_states":nextstates,"values_old":values,
            "next_values_old":nextvalues,"advantages":target['advantage'],
            "lambda_returns":target['lambda_target'],"deltas":target['delta'],
            "old_logps":[s.old_logp for ep in episodes for s in ep],
            "actions":[s.action for ep in episodes for s in ep],
            "rewards":[s.reward for ep in episodes for s in ep],
            "terminated":[s.terminated for ep in episodes for s in ep],
            "truncated":[s.truncated for ep in episodes for s in ep],
            "episodes":len(episodes),"steps":len(states),
            "bootstrap_truncation":bootstrap_truncation,
            "scope":"on_policy_fixed_targets_detached_from_critic"}

def evaluate_policy(actor,critic,gamma,horizon,rollouts=200,seed=2026):
    prob=actor_probabilities(actor)
    exact_continuing=exact_policy_values(prob,gamma)
    exact_success=finite_horizon_success_probability(prob,horizon)
    sampled=collect_rollouts(actor,rollouts,horizon,random.Random(seed))
    success=sum(ep[-1].terminated for ep in sampled)
    truncated=sum(ep[-1].truncated for ep in sampled)
    return {"forward_probabilities":prob,"exact_continuing_value":exact_continuing,
            "exact_success_within_limit":exact_success,
            "sampled_successes":success,"sampled_episodes":rollouts,
            "sampled_success_rate":success/rollouts,"sampled_truncations":truncated,
            "value_rmse":value_error(critic,actor,gamma)['rmse'],
            "note":"固定环境的精确计算＋有限采样；并非模型数学推理能力"}

In [6]:
seed_all(6301)
example_actor = Actor(24).double()
example_critic = Critic(24).double()
example_episodes = collect_rollouts(example_actor, 14, 6, random.Random(63013))
print("实际回合长度：", [len(ep) for ep in example_episodes])
print("实际steps：", sum(len(ep) for ep in example_episodes))
print("成功/截断回合：",sum(ep[-1].terminated for ep in example_episodes),
      sum(ep[-1].truncated for ep in example_episodes))

实际回合长度： [6, 5, 3, 4, 3, 6, 4, 4, 6, 3, 4, 4, 6, 5]
实际steps： 63
成功/截断回合： 12 2


## 3. 完整 PPO-Clip 与 critic 目标不变

策略目标继续采用：

\[
L_{\pi}=-\operatorname{mean}_{t}\min\big(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t\big)
\]

Critic 使用上一课明确区分过的 PPO2 **保守最大平方误差值裁剪**（`clipmax`）变体：

\[
L_V=\tfrac12\operatorname{mean}_{t}\max\Big[(V_\theta-R_t)^2,
\big(V_{old}+\operatorname{clip}(V_\theta-V_{old},-\epsilon_V,+\epsilon_V)-R_t\big)^2\Big]
\]

**这里没有把 clipped prediction MSE 和 clipmax 混为一谈。** 旧值、优势、return 仅作为固定数据，不接受当前 Critic 的反向梯度；Actor entropy 系数仍独立计入目标。

本课的重点不是改变这两个 loss，而是确认将同一批有效 transition 分拆计算时，得到**相同的总损失梯度**。[PPO 源代码](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)

In [7]:
def probability_ratio(new_logp, old_logp):
    require(isinstance(new_logp,torch.Tensor) and isinstance(old_logp,torch.Tensor)
            and new_logp.shape==old_logp.shape and new_logp.ndim==1 and new_logp.numel()>0,
            '新旧log概率需要同形状非空向量')
    require(new_logp.is_floating_point() and old_logp.is_floating_point()
            and new_logp.dtype==old_logp.dtype and new_logp.device==old_logp.device,
            '新旧log概率精度/设备非法')
    require(bool(torch.isfinite(new_logp).all() and torch.isfinite(old_logp).all()),'log概率非有限')
    delta=new_logp-old_logp.detach()
    require(bool((delta.detach().abs()<40).all()),'概率比过大，不静默截断')
    return delta.exp()

def policy_loss(new_logp, old_logp, advantage, epsilon):
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0<epsilon<1,'policy clip范围非法')
    require(isinstance(advantage,torch.Tensor) and advantage.shape==new_logp.shape
            and advantage.dtype==new_logp.dtype and advantage.device==new_logp.device
            and bool(torch.isfinite(advantage).all()),'优势形状/精度/数值非法')
    ratio=probability_ratio(new_logp,old_logp)
    a=advantage.detach()
    plain=ratio*a
    bounded=ratio.clamp(1-epsilon,1+epsilon)*a
    obj=torch.minimum(plain,bounded)
    outside=(ratio>1+epsilon)|(ratio<1-epsilon)
    active=((ratio>1+epsilon)&(a>0))|((ratio<1-epsilon)&(a<0))
    return -obj.mean(),{'ratio':ratio,'surrogate':obj,'plain':plain,
                        'ratio_outside_fraction':float(outside.detach().float().mean()),
                        'active_clip_fraction':float(active.detach().float().mean())}

def value_loss(prediction, old_value, target, epsilon=.2, method='clipmax'):
    """三种可选值损失；clipmax不是原始PPO论文的必选部件。

    plain:        0.5*MSE(pred,target)
    clipmax:      0.5*mean(max(raw_squared, clipped_pred_squared)) (PPO2变体)
    clipped_pred: 0.5*MSE(clipped_prediction,target) (SB3可选实现)
    """
    require(method in {'plain','clipmax','clipped_pred'},'value loss方法不支持')
    require(type(epsilon) in (float,int) and math.isfinite(epsilon) and epsilon>0,'value epsilon无效')
    require(all(isinstance(x,torch.Tensor) for x in (prediction,old_value,target))
            and prediction.ndim==1 and prediction.numel()>0
            and prediction.shape==old_value.shape==target.shape,'value输入必须同形状一维向量')
    require(all(x.is_floating_point() and x.dtype==prediction.dtype and x.device==prediction.device
                and bool(torch.isfinite(x).all()) for x in (prediction,old_value,target)),
            'value设备、dtype或数值非法')
    old=old_value.detach(); goal=target.detach()
    raw_squared=(prediction-goal).square()
    clipped_prediction=old+(prediction-old).clamp(-epsilon,epsilon)
    clipped_squared=(clipped_prediction-goal).square()
    if method=='plain': squared=raw_squared
    elif method=='clipmax': squared=torch.maximum(raw_squared,clipped_squared)
    else: squared=clipped_squared
    return .5*squared.mean(), {
        'raw_half_mse':float((.5*raw_squared.detach()).mean()),
        'clipped_prediction_half_mse':float((.5*clipped_squared.detach()).mean()),
        'max_half_mse':float((.5*torch.maximum(raw_squared.detach(),clipped_squared.detach())).mean()),
        'value_outside_clip_fraction':float(((prediction.detach()-old).abs()>epsilon).float().mean()),
        'method':method}

def normalize_advantages(advantages):
    require(isinstance(advantages,torch.Tensor) and advantages.ndim==1 and advantages.numel()>1
            and advantages.is_floating_point() and bool(torch.isfinite(advantages).all()),
            '归一化优势需要超过一个有限浮点样本')
    a=advantages.detach()
    return (a-a.mean())/(a.std(unbiased=False)+1e-8)

def old_state_kl(old_distribution,new_distribution):
    """仅计算旧样本访问的三状态上的Categorical KL(old||new)的平均。"""
    require(isinstance(old_distribution,torch.Tensor) and isinstance(new_distribution,torch.Tensor)
            and old_distribution.ndim==2 and old_distribution.shape==new_distribution.shape
            and old_distribution.shape[1]==2 and old_distribution.numel()>0,'KL分布形状无效')
    require(all(torch.isfinite(x).all() for x in (old_distribution,new_distribution)),'非有限分布')
    for d in (old_distribution,new_distribution):
        require(bool(torch.allclose(d.logsumexp(-1),torch.zeros_like(d[:,0]),atol=1e-5)),
                'KL需要归一化log概率')
    old=old_distribution.detach()
    return (old.exp()*(old-new_distribution)).sum(-1).mean()

## 4. 打乱策略：每个epoch恰好覆盖每一步一次

对于 rollout 的 `T` 个 transition，构建 `0..T-1` 的索引列表，使用每个 epoch 独立的稳定 seed 打乱，然后按 17 个一组分成 minibatch。

例如 `T=38` 时，**17+17+4** 是合法划分；最后4条也必须训练，不允许为了整齐直接丢掉。每个 epoch 的排序后索引，必须严格等于 `list(range(T))`。

随机 seed 来自 `(master_seed, rollout_id, epoch)` 的稳定摘要；Python 内置 `hash()` 不适合拿来做跨进程可复查的种子。不同 seed 也不保证训练能够跨所有版本与设备逐位复现。

In [8]:
def stable_epoch_seed(master: int, rollout_id: int, epoch: int) -> int:
    require(all(type(x) is int and x>=0 for x in (master, rollout_id, epoch)),"种子与索引必须非负整数")
    return int(sha256(["lesson63", master, rollout_id, epoch])[:8],16)


def shuffled_minibatches(step_count: int, minibatch_size: int, seed: int) -> list[list[int]]:
    require(type(step_count) is int and step_count>0,"没有rollout步骤")
    require(type(minibatch_size) is int and minibatch_size>0,"minibatch size非法")
    rng=random.Random(seed)
    ids=list(range(step_count))
    rng.shuffle(ids)
    return [ids[start:start+minibatch_size] for start in range(0,step_count,minibatch_size)]


def split_microbatches(minibatch: list[int], microbatch_size: int) -> list[list[int]]:
    require(type(microbatch_size) is int and microbatch_size>0,"microbatch size非法")
    require(isinstance(minibatch,list) and bool(minibatch),"minibatch为空")
    require(len(set(minibatch))==len(minibatch),"minibatch有重复索引")
    return [minibatch[i:i+microbatch_size] for i in range(0,len(minibatch),microbatch_size)]


def frozen_signature(buffer: dict) -> str:
    names=("states","actions","old_logps","old_values","returns","advantages",
           "raw_advantages","old_distribution","terminated","truncated")
    frozen_tensors=tensor_hash({name:buffer[name] for name in names})
    meta={k:buffer[k] for k in ("rollout_id","episode_lengths","episode_successes",
                              "episode_truncations","sampled_steps")}
    return sha256([frozen_tensors,meta])


def make_buffer(actor: Actor, critic: Critic, episodes: list[list[Step]], cfg: Config,
                rollout_id: int) -> dict:
    cfg.validate()
    require(type(rollout_id) is int and rollout_id>=0,"rollout ID非法")
    require(len(episodes)==cfg.episodes_per_rollout,"episode数量与配置不同")
    for ep in episodes:validate_episode(ep,cfg.horizon)
    values=compute_targets(episodes,critic,cfg.gamma,cfg.gae_lambda,bootstrap_truncation=True)
    states=torch.tensor(values['states'],dtype=torch.long)
    actions=torch.tensor(values['actions'],dtype=torch.long)
    old_logps=torch.tensor(values['old_logps'],dtype=torch.float64)
    old_values=torch.tensor(values['values_old'],dtype=torch.float64)
    returns=torch.tensor(values['lambda_returns'],dtype=torch.float64)
    raw=torch.tensor(values['advantages'],dtype=torch.float64)
    require(len(raw)>1,"全局优势归一化至少2步")
    normalized=(raw-raw.mean())/(raw.std(unbiased=False)+1e-8)
    with torch.no_grad():
        dist=F.log_softmax(actor(states),dim=-1)
        recalculated=dist.gather(-1,actions.unsqueeze(-1)).squeeze(-1)
    require(torch.allclose(old_logps,recalculated,atol=2e-6,rtol=2e-6),
            "采样时old_logp与当前Actor重算不一致")
    result={"states":states,"actions":actions,"old_logps":old_logps,
            "old_values":old_values,"returns":returns,
            "advantages":normalized.detach().clone(),"raw_advantages":raw,
            "old_distribution":dist.detach().clone(),
            "terminated":torch.tensor(values['terminated'],dtype=torch.bool),
            "truncated":torch.tensor(values['truncated'],dtype=torch.bool),
            "rollout_id":rollout_id,"sampled_steps":len(raw),
            "episode_lengths":[len(ep) for ep in episodes],
            "episode_successes":sum(ep[-1].terminated for ep in episodes),
            "episode_truncations":sum(ep[-1].truncated for ep in episodes),
            "actor_stopped":False,"closed":False,"finished_epochs":0,
            "actor_updates":0,"critic_updates":0,
            "actor_before_hash":tensor_hash(actor.state_dict()),
            "critic_before_hash":tensor_hash(critic.state_dict()),
            "expected_actor_hash":tensor_hash(actor.state_dict()),
            "expected_critic_hash":tensor_hash(critic.state_dict())}
    result['seal']=frozen_signature(result)
    return result


def verify_buffer(actor: Actor, critic: Critic,buffer: dict,cfg: Config,rollout_id: int):
    require(not buffer['closed'],"rollout已经关闭")
    require(buffer['rollout_id']==rollout_id,"不能跨rollout使用旧数据")
    require(buffer['seal']==frozen_signature(buffer),"冻结的rollout数组或元数据被篡改")
    require(buffer['finished_epochs']<cfg.epochs,"已超过计划epoch数")
    require(buffer['expected_actor_hash']==tensor_hash(actor.state_dict()),"Actor在受控循环外变化")
    require(buffer['expected_critic_hash']==tensor_hash(critic.state_dict()),"Critic在受控循环外变化")
    require(all(not buffer[n].requires_grad for n in (
        'old_logps','old_values','returns','advantages','old_distribution')),
        "旧策略、GAE与value target不得持有梯度图")

In [9]:
order = shuffled_minibatches(38, 17, stable_epoch_seed(6301,0,0))
print("minibatch长度:", [len(x) for x in order])
print("每个transition恰好出现一次:", sorted(i for x in order for i in x)==list(range(38)))
print("第一批的microbatch长度:", [len(x) for x in split_microbatches(order[0],5)])
print("尾batch仅4步，micro只有一组:", [len(x) for x in split_microbatches(order[-1],5)])

minibatch长度: [17, 17, 4]
每个transition恰好出现一次: True
第一批的microbatch长度: [5, 5, 5, 2]
尾batch仅4步，micro只有一组: [4]


## 5. 最容易写错的权重：microbatch不应平均再平均

假设 minibatch 有17条，将它拆为 `[5,5,5,2]`。完整 loss 是：

\[
L=\frac1{17}\sum_{t=1}^{17}\ell_t
\]

正确的梯度累积是每一小块先**求和，再除以完整minibatch大小17**：

```python
optimizer.zero_grad(set_to_none=True)
for micro in microbatches:
    (loss_per_row[micro].sum() / len(current_minibatch)).backward()
optimizer.step()
```

错误写法是每个 micro 先取均值，再除以 micro 的个数。最后只有2条的 micro 会被赋予和5条 micro 一样的总权重，改变了目标。

**不执行每个 micro 的 `optimizer.step()`！** 本课一次 minibatch 只更新一次；梯度裁剪也要等全部 micro 累积后再执行。

这里是代数层面的同等目标，浮点计算顺序不同仍可能出现微小舍入误差。

In [10]:
def per_step_losses(actor: Actor,critic: Critic,buffer: dict,index: torch.Tensor,cfg: Config):
    require(index.dtype==torch.long and index.ndim==1 and index.numel()>0,"需要非空long索引")
    s=buffer['states'][index];a=buffer['actions'][index]
    old_lp=buffer['old_logps'][index]
    adv=buffer['advantages'][index].detach()
    old_v=buffer['old_values'][index].detach()
    target=buffer['returns'][index].detach()
    dist=F.log_softmax(actor(s),dim=-1)
    current_lp=dist.gather(-1,a.unsqueeze(-1)).squeeze(-1)
    ratio=probability_ratio(current_lp,old_lp)
    plain=ratio*adv
    clipped=ratio.clamp(1-cfg.clip_ratio,1+cfg.clip_ratio)*adv
    entropy=-(dist.exp()*dist).sum(-1)
    actor_per=-torch.minimum(plain,clipped)-cfg.entropy_coef*entropy
    predicted=critic(s)
    raw_square=(predicted-target).square()
    clipped_values=old_v+(predicted-old_v).clamp(-cfg.value_clip,cfg.value_clip)
    critic_per=.5*torch.maximum(raw_square,(clipped_values-target).square())
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    outside=(ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio)
    return actor_per,critic_per,{
        'ratio':ratio.detach(),'active_clip':active.detach(),'outside':outside.detach(),
        'entropy':entropy.detach(), 'old_minus_new':(old_lp-current_lp).detach(),
        'value_abs_error':(predicted-target).abs().detach()}


@torch.no_grad()
def rollout_diagnostics(actor: Actor,critic: Critic,buffer: dict,cfg: Config) -> dict:
    """整个已观察rollout按step频数统计，而不是平均3个状态或minibatch均值。"""
    actor.eval();critic.eval()
    dist=F.log_softmax(actor(buffer['states']),dim=-1)
    old=buffer['old_distribution'];a=buffer['actions']
    current_lp=dist.gather(-1,a[:,None]).squeeze(-1)
    ratio=probability_ratio(current_lp,buffer['old_logps'])
    old_minus_new=buffer['old_logps']-current_lp
    pointwise=(old.exp()*(old-dist)).sum(-1)
    require(float(pointwise.min())>-1e-8,"归一化分类KL数值异常")
    sample_estimator=((ratio-1)-ratio.log()).mean()  # 非负的近似估计形式
    adv=buffer['advantages']
    out=((ratio>1+cfg.clip_ratio)|(ratio<1-cfg.clip_ratio))
    active=((ratio>1+cfg.clip_ratio)&(adv>0))|((ratio<1-cfg.clip_ratio)&(adv<0))
    ent=-(dist.exp()*dist).sum(-1)
    predicted=critic(buffer['states'])
    result={'transitions':len(a),
        'old_state_kl_exact_visited':float(pointwise.mean()),
        'sampled_approx_kl':float(old_minus_new.mean()),
        'sampled_nonnegative_estimator':float(sample_estimator),
        'sampled_approx_may_be_negative':True,
        'ratio_min':float(ratio.min()), 'ratio_max':float(ratio.max()),
        'ratio_outside_fraction':float(out.double().mean()),
        'active_clip_fraction':float(active.double().mean()),
        'entropy_per_transition':float(ent.mean()),
        'critic_half_mse':float((.5*(predicted-buffer['returns']).square()).mean()),
        'mean_normalized_advantage':float(adv.mean()),
        'old_state_frequency':[int((buffer['states']==s).sum()) for s in range(GOAL)]}
    return result


def full_minibatch_gradients(actor: Actor,critic: Critic,buffer: dict,
                             minibatch: list[int],cfg: Config, microbatch_size: int,
                             compute_actor: bool=True):
    """通过一个采样minibatch中的microchunks累积；此函数不执行step。

    每个micro的loss按该micro行数求和后 / 当前minibatch总行数。
    对actor/critic分别反向，保证最后不满micro时也等价于直接full-minibatch求均值。
    """
    require(type(microbatch_size) is int and microbatch_size>0,'micro大小非法')
    require(bool(minibatch) and len(set(minibatch))==len(minibatch),"minibatch索引重复/空")
    require(all(type(i) is int and 0<=i<buffer['sampled_steps'] for i in minibatch),"索引越界")
    actor.zero_grad(set_to_none=True);critic.zero_grad(set_to_none=True)
    mini_size=len(minibatch)
    parts=split_microbatches(minibatch,microbatch_size)
    sums={'actor_loss_sum':0.,'critic_loss_sum':0.,'entropy_sum':0.,'clip_count':0,
          'outside_count':0,'old_minus_new_sum':0.,'abs_value_error_sum':0.}
    for micro in parts:
        index=torch.tensor(micro,dtype=torch.long)
        actor_per,critic_per,trace=per_step_losses(actor,critic,buffer,index,cfg)
        # Do NOT divide by len(parts) or by len(micro) here.
        if compute_actor:
            (actor_per.sum()/mini_size).backward()
        (critic_per.sum()/mini_size).backward()
        with torch.no_grad():
            sums['actor_loss_sum']+=float(actor_per.detach().sum())
            sums['critic_loss_sum']+=float(critic_per.detach().sum())
            sums['entropy_sum']+=float(trace['entropy'].sum())
            sums['clip_count']+=int(trace['active_clip'].sum())
            sums['outside_count']+=int(trace['outside'].sum())
            sums['old_minus_new_sum']+=float(trace['old_minus_new'].sum())
            sums['abs_value_error_sum']+=float(trace['value_abs_error'].sum())
    sums.update({'size':mini_size,'micro_sizes':[len(p) for p in parts],
                 'actor_loss_before':sums['actor_loss_sum']/mini_size,
                 'critic_loss_before':sums['critic_loss_sum']/mini_size,
                 'entropy_before':sums['entropy_sum']/mini_size,
                 'active_clip_fraction_before':sums['clip_count']/mini_size,
                 'ratio_outside_fraction_before':sums['outside_count']/mini_size,
                 'sampled_approx_kl_before':sums['old_minus_new_sum']/mini_size})
    return sums


def minibatch_update(actor: Actor,critic: Critic,buffer: dict,minibatch: list[int],
                     cfg: Config,actor_optimizer,critic_optimizer,microbatch_size: int) -> dict:
    require(bool(minibatch),"minibatch为空")
    old_diag=rollout_diagnostics(actor,critic,buffer,cfg)
    if not buffer['actor_stopped'] and old_diag['old_state_kl_exact_visited']>cfg.target_kl:
        buffer['actor_stopped']=True
    actor_update=not buffer['actor_stopped']
    actor_opt={id(p) for g in actor_optimizer.param_groups for p in g['params']}
    critic_opt={id(p) for g in critic_optimizer.param_groups for p in g['params']}
    require(actor_opt=={id(p) for p in actor.parameters()},"Actor优化器对象错误")
    require(critic_opt=={id(p) for p in critic.parameters()},"Critic优化器对象错误")
    require(not actor_opt&critic_opt,"Actor/Critic优化器不能共享对象")
    stats=full_minibatch_gradients(actor,critic,buffer,minibatch,cfg,microbatch_size,
                                   compute_actor=actor_update)
    if not actor_update:
        actor.zero_grad(set_to_none=True)
        actor_norm=None
    else:
        actor_norm=float(nn.utils.clip_grad_norm_(actor.parameters(),cfg.grad_clip,error_if_nonfinite=True))
        actor_optimizer.step()
        buffer['actor_updates']+=1
    critic_norm=float(nn.utils.clip_grad_norm_(critic.parameters(),cfg.grad_clip,error_if_nonfinite=True))
    critic_optimizer.step();buffer['critic_updates']+=1
    buffer['expected_actor_hash']=tensor_hash(actor.state_dict())
    buffer['expected_critic_hash']=tensor_hash(critic.state_dict())
    require(buffer['seal']==frozen_signature(buffer),"优化期间旧采样或目标被改变")
    return {'size':stats['size'],'micro_sizes':stats['micro_sizes'],
            'actor_updated':actor_update,'critic_updated':True,
            'actor_stopped':buffer['actor_stopped'],
            'old_state_kl_at_gate':old_diag['old_state_kl_exact_visited'],
            'sampled_kl_at_gate':old_diag['sampled_approx_kl'],
            'actor_grad_norm_before_clip':actor_norm,'critic_grad_norm_before_clip':critic_norm,
            **{k:v for k,v in stats.items() if k in ('actor_loss_before','critic_loss_before',
                'entropy_before','active_clip_fraction_before','ratio_outside_fraction_before',
                'sampled_approx_kl_before','clip_count','outside_count','actor_loss_sum',
                'critic_loss_sum','entropy_sum','old_minus_new_sum')}}

In [11]:
def math_experiments() -> dict:
    # episode平均与step平均回答不同的问题。
    episode_values=[[1.],[0.]*5]
    per_episode=[sum(ep)/len(ep) for ep in episode_values]
    transition_values=[v for ep in episode_values for v in ep]
    micro_values=[1.,1.,0.]
    micro_groups=[micro_values[:2],micro_values[2:]]
    # 单个参数上的精确梯度：每一个transition贡献一个线性样本项 theta*x。
    features=torch.tensor([1.,2.,8.],dtype=torch.float64)
    theta=torch.tensor(1.,dtype=torch.float64,requires_grad=True)
    direct=(theta*features).mean()
    full_grad=torch.autograd.grad(direct,theta,retain_graph=True)[0]
    correct_micro=((theta*features[:2]).sum()/3+(theta*features[2:]).sum()/3)
    correct_grad=torch.autograd.grad(correct_micro,theta,retain_graph=True)[0]
    wrong=((theta*features[:2]).mean()+(theta*features[2:]).mean())/2
    wrong_grad=torch.autograd.grad(wrong,theta)[0]
    raw=torch.tensor([-2.,-1.,0.,1.,2.],dtype=torch.float64)
    global_norm=(raw-raw.mean())/(raw.std(unbiased=False)+1e-8)
    local_norm=torch.cat([normalize_advantages(raw[:2]),normalize_advantages(raw[2:])])
    # p_old(a) high, p_new(a) higher: a single observed old-new logp is negative.
    old=torch.tensor([.8,.2],dtype=torch.float64)
    new=torch.tensor([.9,.1],dtype=torch.float64)
    sampled=float((old.log()-new.log())[0])
    exact=float((old*(old.log()-new.log())).sum())
    # 同一个旧状态的KL分布；状态采样频率9:1，不能均匀平均state ID。
    p_old=torch.tensor([[.5,.5],[.5,.5]],dtype=torch.float64)
    p_new=torch.tensor([[.55,.45],[.95,.05]],dtype=torch.float64)
    state_kl=(p_old*(p_old.log()-p_new.log())).sum(-1)
    weighted=float((state_kl*torch.tensor([.9,.1],dtype=torch.float64)).sum())
    return {
      'episode_vs_transition':{'episode_lengths':[1,5],
          'mean_of_episode_means':sum(per_episode)/2,
          'mean_over_transitions':sum(transition_values)/len(transition_values),
          'note':'两种均值目标不同；episode成功率应以回合为分母，step级PPO指标以transition为分母'},
      'short_tail_minibatch':{'minibatch_lengths':[2,1],
          'unweighted_mean_of_means':sum(sum(a)/len(a) for a in micro_groups)/2,
          'correct_step_mean':sum(micro_values)/len(micro_values)},
      'micro_gradient':{'full_gradient':float(full_grad),
          'correct_weighted_micro_gradient':float(correct_grad),
          'wrong_average_of_micro_gradients':float(wrong_grad),
          'micro_lengths':[2,1]},
      'advantage_normalization':{'raw':raw.tolist(),
          'global':global_norm.tolist(),'per_chunk':local_norm.tolist(),
          'per_chunk_changes_objective':not torch.allclose(global_norm,local_norm)},
      'kl_views':{'sampled_old_minus_new_for_action_0':sampled,
          'exact_old_to_new_2_actions':exact,
          'state0_kl':float(state_kl[0]),'state1_kl':float(state_kl[1]),
          'frequency_weighted_kl':weighted,'uniform_state_kl':float(state_kl.mean()),
          'note':'单样本old-new logp可为负；访问状态上的精确KL非负；访问频率改变均值'},
    }

In [12]:
print(json.dumps(math_experiments()["micro_gradient"], ensure_ascii=False, indent=2))

{
  "full_gradient": 3.6666666666666665,
  "correct_weighted_micro_gradient": 3.6666666666666665,
  "wrong_average_of_micro_gradients": 4.75,
  "micro_lengths": [
    2,
    1
  ]
}


## 6. 真正用神经网络梯度检查微批等价

只比较 loss 数字是不够的。本课构造一份固定 17 transition 的 minibatch，分别用完整批次和 `5+5+5+2` 四个 micro 做：

1. 比较 Actor loss、Critic loss；
2. 比较两套网络所有参数的 `.grad`；
3. 分别执行一次相同配置的 Adam，核对权重。

我们的计算含 entropy、PPO-Clip 和 `clipmax`，不是单独拿一个线性 toy loss 骗过检查。

代码在固定检查中还覆盖短尾 batch、重复索引、索引越界和梯度不因冻结旧数据而泄漏。

In [13]:
demo_cfg = Config(rollouts=2,episodes_per_rollout=14,epochs=2,minibatch_size=17,microbatch_size=5)
demo_buffer = make_buffer(example_actor,example_critic,example_episodes,demo_cfg,0)
mini_indices = list(range(17))
a1,c1 = copy.deepcopy(example_actor),copy.deepcopy(example_critic)
a2,c2 = copy.deepcopy(example_actor),copy.deepcopy(example_critic)
full = full_minibatch_gradients(a1,c1,demo_buffer,mini_indices,demo_cfg,17)
micro = full_minibatch_gradients(a2,c2,demo_buffer,mini_indices,demo_cfg,5)
gap = max(float((p.grad-q.grad).abs().max()) for left,right in ((a1,a2),(c1,c2))
          for p,q in zip(left.parameters(),right.parameters()))
print("micro长度:", micro["micro_sizes"])
print("Actor loss差:", abs(full["actor_loss_before"]-micro["actor_loss_before"]))
print("Critic loss差:", abs(full["critic_loss_before"]-micro["critic_loss_before"]))
print("全部参数梯度最大差:", gap)

micro长度: [5, 5, 5, 2]
Actor loss差: 0.0
Critic loss差: 0.0
全部参数梯度最大差: 5.551115123125783e-17


## 7. 统计日志：什么分母，什么测量时刻？

在一个 epoch 内，minibatch 是依次更新的，所以“各 minibatch 更新前的 loss 再汇总”，并不等于某一瞬间整份 rollout 的 loss。日志必须注明 **before-update per minibatch**。

如果不同 minibatch 大小是 `[17,17,4]`，clip fraction、entropy、近似KL等逐 transition 统计都按实际行数加权：

\[
\overline{x}=\frac{\sum_b n_b\,\overline{x}_b}{\sum_b n_b}
\]

不能把三个 batch 的平均值再简单平均。回合成功率则仍然按 episode 数量报告，不能把它也改成 transition 平均。

此外，要区分**新采样步数**、在多 epoch 中被反复送入训练的**transition presentations**，以及真正的 **Actor/Critic optimizer steps**。如果 Actor 被 KL 停止但 Critic 继续，二者更新次数也会不同。

In [14]:
batch_sizes = [17, 3]
batch_means = [1.0, 0.0]
step_mean = sum(n * mean for n, mean in zip(batch_sizes,batch_means)) / sum(batch_sizes)
step_kl = sum(n * mean for n, mean in zip(batch_sizes,[-1.0,0.0])) / sum(batch_sizes)
print("按step权重得出loss:", step_mean)
print("错误batch等权结果:", sum(batch_means)/len(batch_means))
print("短尾大小:", batch_sizes[-1])
print("KL采样代理可为负:", step_kl)

按step权重得出loss: 0.85
错误batch等权结果: 0.5
短尾大小: 3
KL采样代理可为负: -0.85


## 8. 采样KL与访问状态的精确KL，不是同一个数

对已保存的动作，常见 PPO 近似是：

\[
\widehat{KL}_{sample}=\operatorname{mean}_t[\log\pi_{old}(a_t|s_t)-\log\pi_\theta(a_t|s_t)]
\]

单个样本、甚至有限批平均可为负。对同一状态保留了两动作的**完整旧分布**时，可以计算分类 KL：

\[
KL_{old\|new}(s)=\sum_a \pi_{old}(a|s)\log\frac{\pi_{old}(a|s)}{\pi_\theta(a|s)}\ge0
\]

本课将它按**旧rollout实际访问状态的频率**取平均，作为 `old_state_kl_exact_visited`。这在每个已观察状态上的动作维度精确，但**不遍历所有可能的状态轨迹，也不是全局信赖域**。

另有 `clip_fraction`（比例超区间）与 `active_clip_fraction`（该样本因优势方向进入平坦区域）；二者不要混为一项。KL early stop 也只是阻止后续 Actor 步骤，不会回滚上一笔超限更新。[Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)

In [15]:
table = math_experiments()
print("单动作 old-new logp:", table["kl_views"]["sampled_old_minus_new_for_action_0"])
print("该状态完整动作 KL:", table["kl_views"]["exact_old_to_new_2_actions"])
print("访问状态按频数加权KL:", table["kl_views"]["frequency_weighted_kl"])
print("错误地给各状态同样权重:", table["kl_views"]["uniform_state_kl"])

单动作 old-new logp: -0.11778303565638343
该状态完整动作 KL: 0.0444030075868823
访问状态按频数加权KL: 0.08755921147515819
错误地给各状态同样权重: 0.41769538566878806


## 9. minibatch 训练循环与 KL Actor 停止

默认三路对照：

| 分支 | 一个 optimizer step 处理的数据 | 更新频率 |
|---|---|---|
| `full` | 整份当前 rollout | 每epoch一次 |
| `mini` | 打乱后每17条一组 | 每minibatch一次 |
| `accum` | 仍每17条一组；内部拆成5+5+5+2 | 每minibatch一次 |

所有分支使用同样初始 Actor/Critic 参数、rollout seed 列表、epoch seed 规则。**`mini` 与 `accum` 在相同数值条件下应该等价；`full` 不应该与它们等价，因为优化器步数不同。**

本课每次更新前检查当前 Actor 与当次 rollout 旧分布在全部已观察状态上的 KL。超过阈值时，本rollout后续 Actor 更新停止；Critic 仍按固定 λ-return 完成计划的 minibatch 训练。这是**本课明确选择的门控方式**，不是 PPO 标准唯一实现。

默认每次收集25个变长episode，重复3个epoch。只统计已发生的采样、优化器步数和各分支实际消耗；不会声称三路训练有相同计算量。

In [16]:
def summarise_preupdate_minibatches(rows: list[dict]) -> dict:
    """同一epoch所观察的minibatch，按有效transition总数求平均；短尾不获等权。"""
    require(isinstance(rows,list) and bool(rows),"至少一个minibatch")
    total=sum(r['size'] for r in rows)
    require(total>0,"没有transition")
    keys={'actor_loss':'actor_loss_sum', 'critic_loss':'critic_loss_sum',
          'entropy':'entropy_sum', 'sampled_approx_kl':'old_minus_new_sum'}
    result={k:sum(r[v] for r in rows)/total for k,v in keys.items()}
    result.update({'observed_steps':total, 'minibatches':len(rows),
                   'active_clip_fraction':sum(r['clip_count'] for r in rows)/total,
                   'ratio_outside_fraction':sum(r['outside_count'] for r in rows)/total,
                   'actor_updates':sum(r['actor_updated'] for r in rows),
                   'critic_updates':sum(r['critic_updated'] for r in rows),
                   'short_tail_size':rows[-1]['size'],
                   'note':'minibatch BEFORE-update metrics, weighted by actual transitions; not one simultaneous full-buffer evaluation'})
    return result


def train_arm(cfg: Config,mode: str,schedule: list[int],
              original_actor: Actor,original_critic: Critic):
    require(mode in {'full','mini','accum'},"未知训练对照")
    actor=copy.deepcopy(original_actor);critic=copy.deepcopy(original_critic)
    aopt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    vopt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    history=[];a_updates=c_updates=seen=total_episodes=stopped_rollouts=0
    for rid,seed in enumerate(schedule):
        eps=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(seed))
        buf=make_buffer(actor,critic,eps,cfg,rid)
        n=buf['sampled_steps']
        require(len(set(buf['episode_lengths']))>1 or cfg.episodes_per_rollout==1 or n>0,
                "回合长度异常")
        start_diag=rollout_diagnostics(actor,critic,buf,cfg)
        require(start_diag['old_state_kl_exact_visited']<1e-10,"新rollout的old state KL应为0")
        item={'rollout_id':rid,'episodes':cfg.episodes_per_rollout,'steps':n,
              'episode_lengths':buf['episode_lengths'],
              'terminations':buf['episode_successes'],'truncations':buf['episode_truncations'],
              'advantage_raw_mean':float(buf['raw_advantages'].mean()),
              'advantage_raw_std':float(buf['raw_advantages'].std(unbiased=False)),
              'advantages_normalized_once':True,
              'initial':start_diag,'epochs':[]}
        for epoch in range(cfg.epochs):
            mini_size=n if mode=='full' else cfg.minibatch_size
            order=shuffled_minibatches(n,mini_size,stable_epoch_seed(cfg.seed,rid,epoch))
            check_indices=[i for chunk in order for i in chunk]
            require(sorted(check_indices)==list(range(n)),"每epoch必须恰好覆盖所有transition")
            rows=[]
            for mini in order:
                micro_size=len(mini) if mode in {'full','mini'} else cfg.microbatch_size
                row=minibatch_update(actor,critic,buf,mini,cfg,aopt,vopt,micro_size)
                rows.append(row)
            aggregated=summarise_preupdate_minibatches(rows)
            require(aggregated['observed_steps']==n,"缺失transition或重复计算")
            item['epochs'].append({'epoch':epoch+1,'seed':stable_epoch_seed(cfg.seed,rid,epoch),
                                    'coverage_sha256':sha256(order),'batches':rows,
                                    'weighted':aggregated,
                                    'final_kl':rollout_diagnostics(actor,critic,buf,cfg)})
            buf['finished_epochs']+=1
            # 检查这一个epoch的后验状态仍满足被冻结的合同。
            require(buf['seal']==frozen_signature(buf),"GAE、回报或旧策略在epoch间变化")
        buf['closed']=True
        a_updates+=buf['actor_updates'];c_updates+=buf['critic_updates']
        stopped_rollouts+=int(buf['actor_stopped'])
        total_episodes+=cfg.episodes_per_rollout;seen+=n
        item['actor_updates']=buf['actor_updates'];item['critic_updates']=buf['critic_updates']
        item['actor_stopped']=buf['actor_stopped']
        if rid==0 or (rid+1)%8==0 or rid+1==len(schedule):
            dev=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,60,seed=6306)
            item['dev_observation']={'exact_success_within_limit':dev['exact_success_within_limit'],
                                     'value_rmse':dev['value_rmse']}
        history.append(item)
    return actor,critic,{'mode':mode,'history':history,'rollouts':len(schedule),
                         'episodes':total_episodes,'unique_sampled_steps':seen,
                         'actor_optimizer_steps':a_updates,'critic_optimizer_steps':c_updates,
                         'actor_stopped_rollouts':stopped_rollouts,
                         'transition_presentations':seen*cfg.epochs,
                         'selection':'fixed_final_update_no_test_selection',
                         'minibatch_rule':'full=single_step;mini/accum=17_transition_minis_with_5_transition_micros'}


def save_json(path,value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,
                                    indent=2,allow_nan=False)+'\n',encoding='utf-8')


def experiment(cfg: Config|None=None,root='lesson_63_outputs',mode='all') -> Path:
    cfg=cfg or Config();cfg.validate()
    require(mode in {'all','full','mini','accum'},"未知mode")
    torch.set_num_threads(1);seed_all(cfg.seed)
    # double使累积与单批更新的数值校验更严格；不是GPU混合精度 benchmark。
    init_actor=Actor(cfg.actor_hidden).double();init_critic=Critic(cfg.critic_hidden).double()
    initial_actor=tensor_hash(init_actor.state_dict());initial_critic=tensor_hash(init_critic.state_dict())
    rng=random.Random(cfg.seed+1)
    schedule=[rng.randrange(2**31) for _ in range(cfg.rollouts)]
    modes=['full','mini','accum'] if mode=='all' else [mode]
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    out=root/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    out.mkdir(exist_ok=False)
    manifest={'lesson':63,'version':VERSION,'status':'running','python':platform.python_version(),
              'torch':torch.__version__,'device':'cpu','dtype':'float64','config':asdict(cfg),
              'sources':SOURCES,'limitations':['3_state_2_action_continuing_MDP_external_horizon',
                  'full_rollout_GAE_then_shuffled_transition_minibatches',
                  'actor_PPO_clip_entropy_critic_clipmax_no_shared_base',
                  'no_distributed_rollouts_no_RNN_no_pretrained_LLM',
                  'CPU_double_single_seed_not_GPU_performance_or_algorithm_ranking',
                  'no_exact_resume_state']}
    save_json(out/'manifest.json',manifest)
    try:
        baseline=evaluate_policy(init_actor,init_critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
        report={'baseline':baseline,'schedule_sha256':sha256(schedule),
                'initial_actor_hash':initial_actor,'initial_critic_hash':initial_critic,
                'arms':{},'selection':'fixed_final_iteration'}
        save_json(out/'schedule.json',schedule)
        save_json(out/'math.json',math_experiments())
        for name in modes:
            print('PPO minibatch arm:',name,flush=True)
            actor,critic,training=train_arm(cfg,name,schedule,init_actor,init_critic)
            evaluation=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
            report['arms'][name]={**training,'final':evaluation,
                                 'original_actor_unchanged':tensor_hash(init_actor.state_dict())==initial_actor,
                                 'original_critic_unchanged':tensor_hash(init_critic.state_dict())==initial_critic}
            torch.save({'version':VERSION,'arm':name,'config':asdict(cfg),
                        'actor':{k:v.detach().cpu().clone() for k,v in actor.state_dict().items()},
                        'critic':{k:v.detach().cpu().clone() for k,v in critic.state_dict().items()},
                        'actor_sha256':tensor_hash(actor.state_dict()),
                        'critic_sha256':tensor_hash(critic.state_dict())},out/(name+'.pt'))
        if mode=='all':
            mi=report['arms']['mini'];ac=report['arms']['accum']
            miw=torch.load(out/'mini.pt',weights_only=True,map_location='cpu')
            acw=torch.load(out/'accum.pt',weights_only=True,map_location='cpu')
            largest=max(float((miw[k][n]-acw[k][n]).abs().max()) for k in ('actor','critic')
                        for n in miw[k])
            require(largest<1e-8,'标准minibatch与正确micro累积不等价')
            report['micro_vs_mini']={'max_weight_abs_diff':largest,
                  'actor_optimizer_steps_equal':mi['actor_optimizer_steps']==ac['actor_optimizer_steps'],
                  'critic_optimizer_steps_equal':mi['critic_optimizer_steps']==ac['critic_optimizer_steps'],
                  'scope':'same_frozen_rollouts_shuffled_index_orders_grad_denominator_CUDA_not_verified'}
        save_json(out/'training.json',report)
        save_json(out/'reload.json',reload_run(out))
        manifest['status']='completed';save_json(out/'manifest.json',manifest)
        print('输出目录：',out.resolve(),flush=True)
        return out
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        save_json(out/'manifest.json',manifest)
        raise


def reload_run(folder: Path|str) -> dict:
    folder=Path(folder)
    require(folder.is_dir(),'运行目录不存在')
    report=json.loads((folder/'training.json').read_text(encoding='utf-8'))
    output={}
    for name,row in sorted(report['arms'].items()):
        ck=torch.load(folder/(name+'.pt'),map_location='cpu',weights_only=True)
        require(ck['version']==VERSION and ck['arm']==name,'检查点版本或分支错误')
        cfg=Config(**ck['config']);cfg.validate()
        require(tensor_hash(ck['actor'])==ck['actor_sha256'] and
                tensor_hash(ck['critic'])==ck['critic_sha256'],'权重内容摘要不匹配')
        actor=Actor(cfg.actor_hidden).double().requires_grad_(False).eval()
        critic=Critic(cfg.critic_hidden).double().requires_grad_(False).eval()
        actor.load_state_dict(ck['actor'],strict=True);critic.load_state_dict(ck['critic'],strict=True)
        ev=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
        before=row['final']
        err=max(abs(ev[k]-before[k]) for k in ('exact_success_within_limit','value_rmse'))
        require(err<1e-11 and ev['forward_probabilities']==before['forward_probabilities'],
                '重载指标不一致')
        output[name]={'max_abs_metric_diff':err,
                      'trainable_parameters':sum(p.numel() for p in list(actor.parameters())+
                         list(critic.parameters()) if p.requires_grad)}
    return {'status':'passed','arms':output,'boundary':'trusted_inference_reload_not_exact_resume'}

## 10. 完整CPU训练：先冻结规则，再观察结果

本课已经在制作环境执行完整默认训练；Notebook 运行时会再次从相同种子开始。模型小，所以可以使用 float64 严格核对 micro/mini 差异。

下面的 `RUN_TRAINING=False` 可显式跳过完整训练，数学与独立练习依然可以执行。**跳过训练时，不会制造任何训练报告。**

三路的测试集在最后一步固定后评估，没有依据测试成绩选择步数或分支。它是合成3状态环境，不是新的语言模型。

In [17]:
RUN_DIRECTORY = None
if RUN_TRAINING:
    RUN_DIRECTORY = experiment(cfg, OUTPUT_ROOT, mode="all")
else:
    print("完整训练已跳过，不能声称获得训练结果。")

PPO minibatch arm: full


PPO minibatch arm: mini


PPO minibatch arm: accum


输出目录： /mnt/data/_lesson63_work/notebook/lesson_63_outputs/run_20261103T082359_560652Z


In [18]:
if RUN_DIRECTORY is not None:
    results = json.loads((RUN_DIRECTORY/"training.json").read_text(encoding="utf-8"))
    print("初始限时成功率:", results["baseline"]["exact_success_within_limit"])
    print("模式     限时成功概率  Critic RMSE   Actor更新 / Critic更新    新采样steps")
    for name, entry in results["arms"].items():
        print(f"{name:7} {entry['final']['exact_success_within_limit']:.8f}   "
              f"{entry['final']['value_rmse']:.8f}   "
              f"{entry['actor_optimizer_steps']:4d}/{entry['critic_optimizer_steps']:4d}   "
              f"{entry['unique_sampled_steps']}")
    print("minibatch vs microbatch 的最大权重差:", results.get("micro_vs_mini"))

初始限时成功率: 0.7314319896713004
模式     限时成功概率  Critic RMSE   Actor更新 / Critic更新    新采样steps
accum   1.00000000   0.00006013    323/ 372   1881
full    1.00000000   0.00990057     71/  72   1900
mini    1.00000000   0.00006013    323/ 372   1881
minibatch vs microbatch 的最大权重差: {'actor_optimizer_steps_equal': True, 'critic_optimizer_steps_equal': True, 'max_weight_abs_diff': 1.1463929805444195e-10, 'scope': 'same_frozen_rollouts_shuffled_index_orders_grad_denominator_CUDA_not_verified'}


### 结果应该怎样解释？

**制作环境默认 CPU 运行实测：** 初始六步限时成功概率约0.7314；三路最终均接近1。这仅说明非常小的教学 MDP 易于学会“向前”，不证明泛化推理能力。

`full` 每个 epoch 只有1次更新；`mini`/`accum` 则按17步切分为多个更新，因此它们的最终 Critic RMSE 可以不同。**不能据此说 microbatch 累积比完整批次有更强的学习算法**——更新次数和处理顺序都变了。

`mini` 与 `accum` 在相同调度下最终权重最大差约 `1.15e-10`，更新次数一致。这才是本课真正需要检验的工程等价性。浮点误差非常小但未要求逐位相等，其他设备/版本可能有不同结果。

In [19]:
if RUN_DIRECTORY is not None:
    for name in ("full", "mini", "accum"):
        entry = results["arms"][name]
        lengths = [length for rr in entry["history"] for length in rr["episode_lengths"]]
        print(name, {
            "episode_lengths_seen": sorted(set(lengths)),
            "rollouts_with_actor_stop": entry["actor_stopped_rollouts"],
            "unique_steps": entry["unique_sampled_steps"],
            "step_presentations": entry["transition_presentations"],
            "optimizer_actor_steps": entry["actor_optimizer_steps"],
            "optimizer_critic_steps": entry["critic_optimizer_steps"],
        })

full {'episode_lengths_seen': [3, 4, 5, 6], 'rollouts_with_actor_stop': 1, 'unique_steps': 1900, 'step_presentations': 5700, 'optimizer_actor_steps': 71, 'optimizer_critic_steps': 72}
mini {'episode_lengths_seen': [3, 4, 5, 6], 'rollouts_with_actor_stop': 3, 'unique_steps': 1881, 'step_presentations': 5643, 'optimizer_actor_steps': 323, 'optimizer_critic_steps': 372}
accum {'episode_lengths_seen': [3, 4, 5, 6], 'rollouts_with_actor_stop': 3, 'unique_steps': 1881, 'step_presentations': 5643, 'optimizer_actor_steps': 323, 'optimizer_critic_steps': 372}


## 11. 重载时不能信任旧报告的结论

每次训练保存新目录，包括：

```text
manifest.json       配置、环境、边界与完成状态
schedule.json       三路共享的 rollout seed
math.json           不等长与KL的可手算反例
training.json       每批的变长episode、epoch分组、损失与KL
full.pt             整rollout更新的Actor/Critic
mini.pt             直接minibatch的Actor/Critic
accum.pt            正确micro累积的Actor/Critic
reload.json         从干净网络对象读取权重后重新评估
checks.json         固定检查
mutations.json      故障注入
```

checkpoint包含模型权重与其摘要，用于检测误接或意外变化；**摘要不是数字签名**。重载是推理与指标核对，不是完整优化器、rollout游标、RNG状态的精确断点续训。

跨平台随机性与浮点计算顺序可能不同，CPU double的数值一致不能冒称已验证CUDA性能。[PyTorch autograd与版本差异](https://docs.pytorch.org/docs/stable/notes/autograd.html)

In [20]:
if RUN_DIRECTORY is not None:
    print("独立建网重载核对:")
    print(json.dumps(reload_run(RUN_DIRECTORY), ensure_ascii=False, indent=2))

独立建网重载核对:
{
  "status": "passed",
  "arms": {
    "accum": {
      "max_abs_metric_diff": 0.0,
      "trainable_parameters": 0
    },
    "full": {
      "max_abs_metric_diff": 0.0,
      "trainable_parameters": 0
    },
    "mini": {
      "max_abs_metric_diff": 0.0,
      "trainable_parameters": 0
    }
  },
  "boundary": "trusted_inference_reload_not_exact_resume"
}


## 12. 独立练习：先实现，再运行检查器

练习函数已经有明确的输入/输出约定。检查默认关闭，因此执行全部单元成功不代表你已经写出答案。

**练习 A：按样本数加权平均。** 输入不同 minibatch 的均值和样本数量，返回同一rollout所有 transition 的均值。拒绝不等长、空列表与零分母。

In [21]:
def check_exercise_a(fn) -> dict:
    checks=[]
    for means,sizes,wanted in [([1.,0.],[2,1],2/3),([0.,1.,-1.],[1,3,1],.4),([-2.,8.],[2,2],3.)]:
        require(abs(fn(means,sizes)-wanted)<1e-12,'weighted mean数值错误')
        checks.append('correct_'+str(len(checks)))
    try:fn([1.,2.],[1])
    except (LessonError,ValueError):checks.append('unequal_length_rejected')
    else:raise AssertionError('需要拒绝不等长')
    try:fn([1.,2.],[1,0])
    except (LessonError,ValueError):checks.append('zero_count_rejected')
    else:raise AssertionError('需要拒绝零样本')
    return {'count':len(checks),'checks':checks}


def check_exercise_b(fn) -> dict:
    value=torch.tensor([1.,2.,8.,10.],dtype=torch.float64,requires_grad=True)
    groups=[[0,1],[2],[3]]
    result=fn(value,groups)
    wanted=value.mean()
    require(torch.allclose(result,wanted,atol=1e-12,rtol=1e-12),'数值不等于全batch均值')
    grad=torch.autograd.grad(result,value,retain_graph=True)[0]
    require(torch.allclose(grad,torch.ones_like(value)/4,atol=1e-12),'micro梯度权重错误')
    original=value.detach().clone()
    require(torch.equal(value.detach(),original),'不能修改输入')
    try:fn(value,[[0,1],[1,2],[3]])
    except (LessonError,ValueError):pass
    else:raise AssertionError('重复索引应拒绝')
    return {'status':'passed','gradient':grad.tolist()}


def check_exercise_c(fn) -> dict:
    p=torch.tensor([[.8,.2],[.5,.5]],dtype=torch.float64)
    q=torch.tensor([[.9,.1],[.8,.2]],dtype=torch.float64)
    counts=torch.tensor([9,1],dtype=torch.long)
    got=fn(p.log(),q.log(),counts)
    want=exact_state_kl_reference(p.log(),q.log(),counts)
    require(torch.allclose(got,want,atol=1e-12),'没有按访问频数加权')
    single=(p.log()-q.log())[0,0]
    require(single<0 and got>=0,'样本log比值与全状态KL不应混淆')
    qlogs=q.log().detach().requires_grad_(True)
    gradient=torch.autograd.grad(fn(p.log(),qlogs,counts),qlogs)[0]
    require(torch.isfinite(gradient).all() and gradient.abs().sum()>0,'new分布需要梯度')
    try:fn(p.log(),q.log(),torch.tensor([0,0]))
    except (LessonError,ValueError):pass
    else:raise AssertionError('访问频数不能全零')
    return {'status':'passed','grad_norm':float(gradient.norm())}

In [22]:
RUN_EXERCISE_A = False

def weighted_mean_ex(batch_means, sizes):
    raise NotImplementedError("按实际样本数加权")

if RUN_EXERCISE_A:
    print(check_exercise_a(weighted_mean_ex))

**练习 B：不等长 microbatch 累积。** 输入当前 minibatch 内每条记录的可导损失和全部 micro 索引组，要求每个索引只出现一次，整体结果与直接 `.mean()` 相同，梯度也相同。不要每个 micro 自行取均值。

In [23]:
RUN_EXERCISE_B = False

def micro_loss_ex(per_row_loss, micro_indices):
    raise NotImplementedError("不等长micro的求和除以整个minibatch大小")

if RUN_EXERCISE_B:
    print(check_exercise_b(micro_loss_ex))

**练习 C：访问状态频数加权的精确 KL。** 输入正规化的旧/新状态-动作 log 分布与各状态访问次数。当前分布保留梯度，旧分布固定；结果是一个可导标量，不是单样本 old-new log 概率。

In [24]:
RUN_EXERCISE_C = False

def exact_state_kl_ex(old_logdist, new_logdist, visit_counts):
    raise NotImplementedError("状态KL乘以访问频数比例")

if RUN_EXERCISE_C:
    print(check_exercise_c(exact_state_kl_ex))

## 13. 参考答案

先完成独立练习，再展开下一单元；参考函数使用不同名称，不覆盖练习区。数值和梯度都会检查。

In [25]:
def weighted_mean_reference(batch_means: list[float], sizes: list[int]) -> float:
    require(type(batch_means) is list and type(sizes) is list and len(batch_means)==len(sizes)>0,
            '需要等长非空列表')
    require(all(type(x) in (float,int) and math.isfinite(x) for x in batch_means),
            '均值应为有限数值')
    require(all(type(x) is int and x>0 for x in sizes),'分母必须为正整数')
    return sum(m*n for m,n in zip(batch_means,sizes))/sum(sizes)


def micro_loss_reference(per_row_loss: torch.Tensor,micro_indices: list[list[int]]) -> torch.Tensor:
    require(per_row_loss.ndim==1 and per_row_loss.numel()>0,'需要一维非空loss')
    require(bool(torch.isfinite(per_row_loss).all()),'loss必须有限')
    indices=[i for group in micro_indices for i in group]
    require(sorted(indices)==list(range(len(per_row_loss))),'micro必须不重不漏覆盖当前minibatch')
    total=per_row_loss.new_zeros(())
    for group in micro_indices:
        total=total+per_row_loss[group].sum()/len(per_row_loss)
    return total


def exact_state_kl_reference(old_logdist: torch.Tensor,new_logdist: torch.Tensor,
                             visit_counts: torch.Tensor) -> torch.Tensor:
    require(old_logdist.shape==new_logdist.shape and old_logdist.ndim==2,
            '分布维度错误')
    require(visit_counts.ndim==1 and visit_counts.shape[0]==old_logdist.shape[0],
            '访问频数维度错误')
    require(bool((visit_counts>=0).all()) and float(visit_counts.sum())>0,
            '频数需要非负且总和正')
    require(bool(torch.isfinite(old_logdist).all() and torch.isfinite(new_logdist).all()),
            'KL分布必须有限')
    require(bool(torch.allclose(old_logdist.logsumexp(-1),torch.zeros_like(old_logdist[:,0]),atol=1e-8))
            and bool(torch.allclose(new_logdist.logsumexp(-1),torch.zeros_like(new_logdist[:,0]),atol=1e-8)),
            '输入必须是正规化log概率')
    per_state=(old_logdist.detach().exp()*(old_logdist.detach()-new_logdist)).sum(-1)
    weights=visit_counts.to(per_state)/visit_counts.sum()
    return (per_state*weights).sum()

In [26]:
print("参考A:", check_exercise_a(weighted_mean_reference))
print("参考B:", check_exercise_b(micro_loss_reference))
print("参考C:", check_exercise_c(exact_state_kl_reference))

参考A: {'count': 5, 'checks': ['correct_0', 'correct_1', 'correct_2', 'unequal_length_rejected', 'zero_count_rejected']}
参考B: {'status': 'passed', 'gradient': [0.25, 0.25, 0.25, 0.25]}
参考C: {'status': 'passed', 'grad_norm': 0.745519952784632}


## 14. 完整检查与真实训练中的冻结审计

自动检查不仅验证小公式，还验证**真实网络所有参数的梯度**和一次Adam更新的等价性：

- 全 rollout 只归一化一次 GAE，旧值、return、old logp固定。
- 每epoch打乱并不重不漏覆盖每个transition。
- 最后短 minibatch 被正确计入损失和日志。
- Actor 的 KL 超限只停止本rollout后续 Actor 更新；Critic 继续且不能让Actor获得残余梯度。
- `mini` 和 `accum` 真实多轮训练结果数值一致。
- 保存的对象从磁盘恢复后，Actor概率、Critic状态值一致。

这些是本课的工程契约测试，**不是通用 PPO、安全性或强化学习算法优越性认证**。

In [27]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError('检查失败：'+name)
        self.names.append(name)
    def close(self,name,a,b,atol=1e-10,rtol=1e-10):
        aa=torch.as_tensor(a,dtype=torch.float64);bb=torch.as_tensor(b,dtype=torch.float64)
        self.ok(name,aa.shape==bb.shape and bool(torch.allclose(aa,bb,atol=atol,rtol=rtol)))
    def raises(self,name,fn,expected=(LessonError,ValueError)):
        try:fn()
        except expected:self.names.append(name)
        else:raise AssertionError('没有拒绝：'+name)


def run_checks() -> dict:
    torch.set_num_threads(1);seed_all(63010)
    c=Checks();math_table=math_experiments()
    c.close('回合等权均值',math_table['episode_vs_transition']['mean_of_episode_means'],.5)
    c.close('transition等权均值',math_table['episode_vs_transition']['mean_over_transitions'],1/6)
    c.close('尾batch错误均值',math_table['short_tail_minibatch']['unweighted_mean_of_means'],.5)
    c.close('尾batch正确均值',math_table['short_tail_minibatch']['correct_step_mean'],2/3)
    c.close('全batch梯度=正确micro',math_table['micro_gradient']['full_gradient'],
            math_table['micro_gradient']['correct_weighted_micro_gradient'])
    c.ok('错误平均micro梯度会变化',abs(math_table['micro_gradient']['wrong_average_of_micro_gradients']-
                                         math_table['micro_gradient']['full_gradient'])>1)
    c.ok('每个micro独立归一化改变训练信号',math_table['advantage_normalization']['per_chunk_changes_objective'])
    c.ok('单样本近似KL可以为负',math_table['kl_views']['sampled_old_minus_new_for_action_0']<0)
    c.ok('旧分布到新分布精确KL非负',math_table['kl_views']['exact_old_to_new_2_actions']>0)
    c.ok('按访问频率与按不同state等权KL不同',math_table['kl_views']['uniform_state_kl']>
         math_table['kl_views']['frequency_weighted_kl']*3)
    c.close('练习A参考正确',weighted_mean_reference([1.,0.],[2,1]),2/3)
    c.close('练习B参考梯度不改变',micro_loss_reference(torch.tensor([1.,2.,8.,10.]),[[0,1],[2],[3]]),5.25)
    c.close('练习C参考状态KL',exact_state_kl_reference(
        torch.tensor([[.8,.2],[.5,.5]],dtype=torch.float64).log(),
        torch.tensor([[.9,.1],[.8,.2]],dtype=torch.float64).log(),torch.tensor([9,1])),
        .9*(.8*math.log(.8/.9)+.2*math.log(.2/.1))+.1*(.5*math.log(.5/.8)+.5*math.log(.5/.2)),atol=2e-8)
    c.raises('拒绝负配置回合数',lambda:Config(episodes_per_rollout=0).validate())
    c.raises('拒绝minibatch大小0',lambda:Config(minibatch_size=0).validate())
    c.raises('拒绝micro大于mini',lambda:Config(minibatch_size=8,microbatch_size=9).validate())
    c.raises('拒绝NaN阈值',lambda:Config(target_kl=float('nan')).validate())
    c.raises('拒绝错误clip比',lambda:Config(clip_ratio=1.).validate())
    c.raises('拒绝bool冒充epochs',lambda:Config(epochs=True).validate())
    for a in (0,-1,1.1):
        c.raises('拒绝非法gamma:'+str(a),lambda a=a:Config(gamma=a).validate())

    cfg=Config(rollouts=2,episodes_per_rollout=14,epochs=2,minibatch_size=17,microbatch_size=5)
    c.ok('稳定种子与循环无关',stable_epoch_seed(cfg.seed,2,3)==stable_epoch_seed(cfg.seed,2,3))
    c.ok('每epoch种子不同',stable_epoch_seed(cfg.seed,0,0)!=stable_epoch_seed(cfg.seed,0,1))
    c.raises('拒绝非法随机索引',lambda:stable_epoch_seed(cfg.seed,-1,0))
    indices=shuffled_minibatches(38,17,stable_epoch_seed(cfg.seed,1,0))
    c.ok('38步拆成17_17_4',[len(p) for p in indices]==[17,17,4])
    c.ok('shuffle每步不重不漏',sorted(x for p in indices for x in p)==list(range(38)))
    c.ok('同seed相同顺序',indices==shuffled_minibatches(38,17,stable_epoch_seed(cfg.seed,1,0)))
    c.ok('换seed确实改动顺序',indices!=shuffled_minibatches(38,17,stable_epoch_seed(cfg.seed,1,1)))
    c.ok('mini17拆micro5的尾块',[len(p) for p in split_microbatches(list(range(17)),5)]==[5,5,5,2])
    c.ok('尾mini4仍保留',split_microbatches(list(range(4)),5)==[list(range(4))])
    c.raises('拒绝micro重复索引',lambda:split_microbatches([0,1,1],2))
    c.raises('拒绝空micro',lambda:split_microbatches([],2))
    c.raises('拒绝非法总数',lambda:shuffled_minibatches(0,17,0))
    c.close('练习A三种验收',check_exercise_a(weighted_mean_reference)['count'],5)
    c.ok('练习B验收成功',check_exercise_b(micro_loss_reference)['status']=='passed')
    c.ok('练习C验收成功',check_exercise_c(exact_state_kl_reference)['status']=='passed')

    # 数据环境继承第62课：真实终止与外部cut的bootstrap分开。
    nxt,r,term,trunc=transition(2,1,1,6)
    c.ok('到达终点是真终止',nxt==3 and term and not trunc)
    nxt,r,term,trunc=transition(1,0,5,6)
    c.ok('外部horizon不等于真实终止',nxt==1 and trunc and not term)
    ep=[Step(0,1,-.01,1,False,False,math.log(.5)),
        Step(1,0,-.01,1,False,True,math.log(.5))]
    # 单独用generalized_advantage_estimation测试trace边界，不要求它是完整horizon episode。
    gae=generalized_advantage_estimation([ep],[.2,.3],[.3,.4],.95,.95)
    c.close('截断仍保留下一状态bootstrap',gae['delta'][-1],-.01+.95*.4-.3)
    c.close('截断trace停止时最后一条GAE=delta',gae['advantage'][-1],gae['delta'][-1])
    c.raises('回合终止后仍有下一条无效',lambda:generalized_advantage_estimation(
        [[Step(2,1,.99,3,True,False,math.log(.5)),ep[1]]],[0.,0.],[0.,0.],.95,.95))
    c.ok('不同长度合法episode可以拼到一个buffer',len([ep[:1],ep])==2)

    actor=Actor(cfg.actor_hidden).double().eval()
    critic=Critic(cfg.critic_hidden).double().eval()
    episodes=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(63012))
    for epi in episodes:validate_episode(epi,cfg.horizon)
    buffer=make_buffer(actor,critic,episodes,cfg,0)
    c.ok('确实得到不同episode长度',len(set(buffer['episode_lengths']))>1)
    c.ok('steps等于长度之和',sum(buffer['episode_lengths'])==buffer['sampled_steps'])
    c.ok('回合计数等于14',len(buffer['episode_lengths'])==14)
    c.ok('结尾必然是终止或外部截断',buffer['episode_successes']+buffer['episode_truncations']==14)
    c.ok('冻结旧分布形状',buffer['old_distribution'].shape==(buffer['sampled_steps'],2))
    c.ok('每transition拥有一个固定GAE',buffer['advantages'].shape==(buffer['sampled_steps'],))
    c.close('rollout全局优势均值为0',float(buffer['advantages'].mean()),0.,atol=1e-9)
    c.ok('rollout优势归一化不是每条episode',len(set(buffer['episode_lengths']))>1)
    c.close('初始全rollout精确KL=0',rollout_diagnostics(actor,critic,buffer,cfg)['old_state_kl_exact_visited'],0.,atol=1e-12)
    c.close('初始样本近似KL=0',rollout_diagnostics(actor,critic,buffer,cfg)['sampled_approx_kl'],0.,atol=2e-6)
    c.ok('旧分布逐行正规化',bool(torch.allclose(buffer['old_distribution'].logsumexp(-1),
                          torch.zeros(buffer['sampled_steps'],dtype=torch.float64))))
    c.ok('旧value和返回目标没有梯度',not any(buffer[name].requires_grad for name in
                                      ('old_logps','returns','old_values','advantages')))
    before=frozen_signature(buffer)
    c.raises('初始错误rollout ID拒绝',lambda:verify_buffer(actor,critic,buffer,cfg,1))
    c.ok('未修改合法rollout核对通过',verify_buffer(actor,critic,buffer,cfg,0) is None)
    for key,delta in [('old_logps',.1),('old_values',.1),('returns',.1),('advantages',.1),
                      ('old_distribution',.1)]:
        changed=copy.deepcopy(buffer)
        changed[key][0]+=delta
        c.raises('封存数组被改:'+key,lambda b=changed:verify_buffer(actor,critic,b,cfg,0))
    changed=copy.deepcopy(buffer);changed['episode_lengths'][0]+=1
    c.raises('封存episode长度不能改',lambda:verify_buffer(actor,critic,changed,cfg,0))
    with torch.no_grad():next(actor.parameters()).add_(.01)
    c.raises('检查Actor外部变化',lambda:verify_buffer(actor,critic,buffer,cfg,0))
    actor=Actor(cfg.actor_hidden).double().eval();critic=Critic(cfg.critic_hidden).double().eval()
    # 重建单独buffer：sampler与actor必须同一参数快照。
    episodes=collect_rollouts(actor,cfg.episodes_per_rollout,cfg.horizon,random.Random(63013))
    buffer=make_buffer(actor,critic,episodes,cfg,0)
    with torch.no_grad():next(critic.parameters()).add_(.01)
    c.raises('检查Critic外部变化',lambda:verify_buffer(actor,critic,buffer,cfg,0))
    critic=Critic(cfg.critic_hidden).double().eval()
    buffer=make_buffer(actor,critic,episodes,cfg,0)
    c.ok('封存状态校验',buffer['seal']==frozen_signature(buffer))

    # 完整mini vs 不等长micro：梯度与一次优化步一致，不按micro数量平摊。
    mini=list(range(min(17,buffer['sampled_steps'])))
    first_a=copy.deepcopy(actor);first_v=copy.deepcopy(critic)
    micro_a=copy.deepcopy(actor);micro_v=copy.deepcopy(critic)
    full_stats=full_minibatch_gradients(first_a,first_v,buffer,mini,cfg,len(mini))
    micro_stats=full_minibatch_gradients(micro_a,micro_v,buffer,mini,cfg,5)
    c.ok('17条mini有5_5_5_2micro',micro_stats['micro_sizes']==[5,5,5,2])
    c.close('Actor loss分解数值等价',full_stats['actor_loss_before'],micro_stats['actor_loss_before'],atol=1e-12)
    c.close('Critic loss分解数值等价',full_stats['critic_loss_before'],micro_stats['critic_loss_before'],atol=1e-12)
    for (name,p),(name2,q) in zip(first_a.named_parameters(),micro_a.named_parameters()):
        c.ok('Actor参数列表相同:'+name,name==name2)
        c.close('Actor micro梯度等价:'+name,p.grad,q.grad,atol=1e-10)
    for (name,p),(name2,q) in zip(first_v.named_parameters(),micro_v.named_parameters()):
        c.ok('Critic参数列表相同:'+name,name==name2)
        c.close('Critic micro梯度等价:'+name,p.grad,q.grad,atol=1e-10)
    for net_a,net_b in ((first_a,micro_a),(first_v,micro_v)):
        oa=torch.optim.Adam(net_a.parameters(),lr=.01)
        ob=torch.optim.Adam(net_b.parameters(),lr=.01)
        oa.step();ob.step()
        for (name,p),(name2,q) in zip(net_a.named_parameters(),net_b.named_parameters()):
            c.close('一次Adam更新累积等价:'+name,p.detach(),q.detach(),atol=1e-10)
    small=list(range(3))
    tail_a=copy.deepcopy(actor);tail_v=copy.deepcopy(critic)
    stats_tail=full_minibatch_gradients(tail_a,tail_v,buffer,small,cfg,5)
    c.ok('不足micro size时仍处理尾部',stats_tail['micro_sizes']==[3])
    c.raises('重复索引拒绝',lambda:full_minibatch_gradients(actor,critic,buffer,[1,1],cfg,2))
    c.raises('越界索引拒绝',lambda:full_minibatch_gradients(actor,critic,buffer,[999],cfg,2))
    c.raises('micro大小0拒绝',lambda:full_minibatch_gradients(actor,critic,buffer,mini,cfg,0))

    # 加权汇总与短尾：step数据只算一次。
    def fake(size,value):
        return {'size':size,'actor_loss_sum':value*size,'critic_loss_sum':value*size,
                'entropy_sum':.5*size,'clip_count':size if value else 0,'outside_count':size if value else 0,
                'old_minus_new_sum':-value*size,'actor_updated':True,'critic_updated':True}
    reported=summarise_preupdate_minibatches([fake(17,1),fake(3,0)])
    c.close('短尾诊断17/20权重',reported['actor_loss'],17/20)
    c.close('尾batch clipfraction按样本加权',reported['active_clip_fraction'],17/20)
    c.close('尾batch KL按样本加权且可负',reported['sampled_approx_kl'],-17/20)
    c.ok('尾batch数量单独报告',reported['short_tail_size']==3)

    # Actor early stop只暂停Actor，Critic可继续在固定target上学习。
    stopped_actor=copy.deepcopy(actor);stopped_critic=copy.deepcopy(critic)
    stop_cfg=Config(rollouts=2,episodes_per_rollout=14,epochs=2,
                    target_kl=1e-9,minibatch_size=17,microbatch_size=5)
    stop_buffer=make_buffer(stopped_actor,stopped_critic,episodes,stop_cfg,0)
    with torch.no_grad():stopped_actor.net[-1].bias[1]+=4.
    stop_buffer['expected_actor_hash']=tensor_hash(stopped_actor.state_dict())
    a_before=tensor_hash(stopped_actor.state_dict())
    v_before=tensor_hash(stopped_critic.state_dict())
    ao=torch.optim.Adam(stopped_actor.parameters(),lr=.01)
    vo=torch.optim.Adam(stopped_critic.parameters(),lr=.01)
    after_stop=minibatch_update(stopped_actor,stopped_critic,stop_buffer,mini,stop_cfg,ao,vo,5)
    c.ok('精确KL阈值触发',after_stop['old_state_kl_at_gate']>stop_cfg.target_kl)
    c.ok('Actor确实停止但Critic更新',not after_stop['actor_updated'] and after_stop['critic_updated'])
    c.ok('Actor冻结参数与梯度',a_before==tensor_hash(stopped_actor.state_dict()) and
         all(p.grad is None for p in stopped_actor.parameters()))
    c.ok('Critic本步实际更新',v_before!=tensor_hash(stopped_critic.state_dict()))
    c.ok('批次不恢复Actor状态',stop_buffer['actor_stopped'])
    with torch.no_grad():stopped_actor.net[-1].bias[1]-=4.
    stop_buffer['expected_actor_hash']=tensor_hash(stopped_actor.state_dict())
    next_stop=minibatch_update(stopped_actor,stopped_critic,stop_buffer,mini,stop_cfg,ao,vo,5)
    c.ok('Actor停止后不自动重新启动',not next_stop['actor_updated'])
    c.ok('Critic停止后仍完成第二步',next_stop['critic_updated'])
    stop_buffer['closed']=True
    c.raises('关闭的rollout不能继续epoch',lambda:verify_buffer(stopped_actor,stopped_critic,stop_buffer,stop_cfg,0))

    # 真实同调度训练，确认 mini 与 accum 数值上等价，不声称full可与mini等价。
    tiny_cfg=Config(rollouts=2,episodes_per_rollout=12,epochs=2,
                    minibatch_size=17,microbatch_size=5)
    seed_all(63011)
    init_a=Actor(tiny_cfg.actor_hidden).double();init_v=Critic(tiny_cfg.critic_hidden).double()
    schedule=[10063,20063]
    aa,av,ar=train_arm(tiny_cfg,'accum',schedule,init_a,init_v)
    ma,mv,mr=train_arm(tiny_cfg,'mini',schedule,init_a,init_v)
    c.close('两分支Actor最终参数向量等价',torch.cat([p.flatten() for p in aa.parameters()]),
            torch.cat([p.flatten() for p in ma.parameters()]),atol=2e-9)
    c.close('两分支Critic最终参数向量等价',torch.cat([p.flatten() for p in av.parameters()]),
            torch.cat([p.flatten() for p in mv.parameters()]),atol=2e-9)
    c.ok('两分支实际更新次数一致',ar['actor_optimizer_steps']==mr['actor_optimizer_steps']
         and ar['critic_optimizer_steps']==mr['critic_optimizer_steps'])
    c.ok('累计未改变观察到的transition数',ar['unique_sampled_steps']==mr['unique_sampled_steps'])
    c.ok('每epoch观察数为rollout步数',all(ep['weighted']['observed_steps']==row['steps']
         for row in ar['history'] for ep in row['epochs']))
    c.ok('所有短尾无样本丢失',all(sum(b['size'] for b in ep['batches'])==row['steps']
         for row in ar['history'] for ep in row['epochs']))
    c.ok('同一epoch指标汇总按step加权',all(ep['weighted']['minibatches']==len(ep['batches'])
         for row in ar['history'] for ep in row['epochs']))
    c.ok('每epoch拥有可复查的打乱seed',all(type(ep['seed']) is int
         for row in ar['history'] for ep in row['epochs']))
    with tempfile.TemporaryDirectory() as tmp:
        a_path=Path(tmp)/'model.pt'
        torch.save({'actor':ma.state_dict(),'critic':mv.state_dict()},a_path)
        saved=torch.load(a_path,weights_only=True,map_location='cpu')
        xr=Actor(tiny_cfg.actor_hidden).double().requires_grad_(False)
        xv=Critic(tiny_cfg.critic_hidden).double().requires_grad_(False)
        xr.load_state_dict(saved['actor']);xv.load_state_dict(saved['critic'])
        c.close('独立权重磁盘往返后Actor概率一致',actor_probabilities(xr),actor_probabilities(ma),atol=0,rtol=0)
        c.close('独立权重磁盘往返后Critic值一致',value_prediction(xv),value_prediction(mv),atol=0,rtol=0)
    return {'passed':len(c.names),'checks':c.names,
            'scope':'exact_micro_gradient_and_cpu_actor_critic_minibatch_not_pretrained_LLM_or_distributed_PPO'}

In [28]:
CHECK_REPORT = run_checks()
print("固定检查通过：", CHECK_REPORT["passed"], "项")
print("范围：", CHECK_REPORT["scope"])
if RUN_DIRECTORY is not None:
    save_json(RUN_DIRECTORY/"checks.json", CHECK_REPORT)

固定检查通过： 125 项
范围： exact_micro_gradient_and_cpu_actor_critic_minibatch_not_pretrained_LLM_or_distributed_PPO


## 15. 故障注入：检查必须真的发现写错的实现

本课主动尝试七类错误：batch等权平均、micro均值平均、忽略状态访问频数、episode均值混作step均值、每个mini再次归一GAE、把负的采样KL判为无效、每epoch不重新打乱。只有对应检查确实抛出错误，才记录为已检出。

这是有针对性的回归测试，不是完整形式验证。

In [29]:
def mutation_checks() -> dict:
    cases=[]
    def detect(name,run):
        try:run()
        except (LessonError,AssertionError,ValueError):
            cases.append({'fault':name,'detected':True})
        else:raise AssertionError('未发现错误：'+name)

    def wrong_avg(means,sizes):
        return sum(means)/len(means)
    detect('把不同大小batch等权平均',lambda:check_exercise_a(wrong_avg))

    def wrong_micro(loss,indices):
        return sum(loss[part].mean() for part in indices)/len(indices)
    detect('将不等长micro的均值再简单平均',lambda:check_exercise_b(wrong_micro))

    def wrong_state_kl(old,new,counts):
        p=old.exp()
        return (p*(old-new)).sum(-1).mean()
    detect('状态KL忽略访问频数',lambda:check_exercise_c(wrong_state_kl))

    math=math_experiments()
    def reject_unweighted_episode_mean():
        require(abs(math['episode_vs_transition']['mean_over_transitions']-
                    math['episode_vs_transition']['mean_of_episode_means'])<1e-10,
                'episode等权不是transition等权')
    detect('误把episode均值当step均值',reject_unweighted_episode_mean)

    def reject_local_advantage_normalization():
        require(not math['advantage_normalization']['per_chunk_changes_objective'],
                '每个mini单独归一化改变了固定优势')
    detect('在每个minibatch重新归一化GAE',reject_local_advantage_normalization)

    def reject_negative_kl_as_invalid():
        require(math['kl_views']['sampled_old_minus_new_for_action_0']>=0,
                '单样本old-new log概率可以为负')
    detect('负的采样KL代理直接判为实现错误',reject_negative_kl_as_invalid)

    def reject_same_order_every_epoch():
        p=shuffled_minibatches(38,17,stable_epoch_seed(6301,0,0))
        q=shuffled_minibatches(38,17,stable_epoch_seed(6301,0,1))
        require(p==q,'每个epoch必须重新打乱步级索引')
    detect('每轮复用同一索引顺序',reject_same_order_every_epoch)

    return {'count':len(cases),'cases':cases,
            'scope':'targeted_function_and_invariant_faults_not_formal_verification'}

In [30]:
MUTATIONS = mutation_checks()
for row in MUTATIONS["cases"]:
    print(row["fault"], "→ 检出：", row["detected"])
print("合计检出：", MUTATIONS["count"])
if RUN_DIRECTORY is not None:
    save_json(RUN_DIRECTORY/"mutations.json", MUTATIONS)

把不同大小batch等权平均 → 检出： True
将不等长micro的均值再简单平均 → 检出： True
状态KL忽略访问频数 → 检出： True
误把episode均值当step均值 → 检出： True
在每个minibatch重新归一化GAE → 检出： True
负的采样KL代理直接判为实现错误 → 检出： True
每轮复用同一索引顺序 → 检出： True
合计检出： 7


## 16. 独立脚本运行与预先规定的新实验

```bash
# 检查和手算，不创建模型训练结果
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --checks-only
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --math-only

# 完整三路；默认minibatch=17，microbatch=5
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py

# 只训练其中一种
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --mode accum

# 单独设计的新开发实验，不覆盖原结果
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --epochs 2 --minibatch-size 13 --microbatch-size 4

# 从某次已保存的模型检查点重建模型并核对指标
python lesson_63_ppo_minibatch_accumulation_kl_diagnostics.py --reload-run lesson_63_outputs/实际目录
```

更改 minibatch 大小、更新次数和学习率后，实验计算量已经变化。请把所有训练配置、观察步数、优化器调用与测试数据身份保存在报告中，不要在查看测试结果后反复调整再称测试“未使用”。

上一课完整 Actor/Critic 损失已有多种实现，本课固定采用 `clipmax`，专门比较批处理与优化器语义。不要把不同价值损失的实验混为仅仅换了 batch size。

In [31]:
def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument('--checks-only',action='store_true')
    mode.add_argument('--math-only',action='store_true')
    mode.add_argument('--reload-run',type=Path)
    parser.add_argument('--mode',choices=['all','full','mini','accum'],default='all')
    parser.add_argument('--rollouts',type=int,default=24)
    parser.add_argument('--episodes-per-rollout',type=int,default=25)
    parser.add_argument('--epochs',type=int,default=3)
    parser.add_argument('--minibatch-size',type=int,default=17)
    parser.add_argument('--microbatch-size',type=int,default=5)
    parser.add_argument('--target-kl',type=float,default=.06)
    parser.add_argument('--output-dir',default='lesson_63_outputs')
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2));return 0
    if args.math_only:
        print(json.dumps(math_experiments(),ensure_ascii=False,indent=2));return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['passed']}项。真实CPU小模型与minibatch，不是预训练LLM成绩。")
    if args.checks_only:return 0
    cfg=Config(rollouts=args.rollouts,episodes_per_rollout=args.episodes_per_rollout,
               epochs=args.epochs,minibatch_size=args.minibatch_size,
               microbatch_size=args.microbatch_size,target_kl=args.target_kl)
    cfg.validate()
    output=experiment(cfg,args.output_dir,args.mode)
    save_json(output/'checks.json',checks)
    save_json(output/'mutations.json',mutation_checks())
    report=json.loads((output/'training.json').read_text(encoding='utf-8'))
    print('基线：',report['baseline']['exact_success_within_limit'],
          'value_RMSE：',report['baseline']['value_rmse'])
    for name,item in report['arms'].items():
        print(name,'精确限时成功率=',item['final']['exact_success_within_limit'],
              'critic RMSE=',item['final']['value_rmse'],
              '更新次数=',item['actor_optimizer_steps'],'/',item['critic_optimizer_steps'])
    print('故障检出：',len(json.loads((output/'mutations.json').read_text())['cases']))
    return 0

## 17. 本课验收与下一课

能够独立解释：

1. minibatch `[17,17,4]` 为什么不能把三个均值等权平均？
2. `5+5+5+2` 的 micro 应该如何缩放梯度？为什么梯度累积只调用一次 `optimizer.step()`？
3. 为什么换一个epoch需要重新打乱**transition索引**，又不能重新计算旧策略、GAE和return？
4. `sampled_approx_kl=-0.05` 为什么不直接违反 KL 的非负性？
5. 访问状态上的精确KL与整个MDP的策略KL有什么差别？
6. Actor 因 KL 停止但 Critic 继续，为何两种更新次数不同？
7. 为什么完整batch与minibatch训练结果不同，并不能说明梯度累积失效？

**下一课第 64 课：PPO 训练稳定性与可复现实验协议。** 在完成了batch与目标的正确性后，研究多随机种子、置信区间、超参数选择和报告规范，避免单次好看的曲线冒充稳健结论。

## 官方资料（依据公开算法与实现核对）

1. [Schulman 等，PPO原论文](https://arxiv.org/abs/1707.06347)：clip代理目标。
2. [OpenAI Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)：目标裁剪与KL停止；裁剪不保证硬信赖域。
3. [Spinning Up PyTorch PPO代码](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)：旧log概率、优势与采样KL诊断。
4. [Stable Baselines3 2.7.1 PPO](https://stable-baselines3.readthedocs.io/en/v2.7.1/_modules/stable_baselines3/ppo/ppo.html)：minibatch、尾批与优势归一化的实现差异。
5. [Schulman 等，GAE](https://arxiv.org/abs/1506.02438)：TD残差、λ-return。
6. [Gymnasium：Termination 与 Truncation](https://gymnasium.farama.org/v0.26.3/tutorials/handling_time_limits/)：截断仍需bootstrap。
7. [PyTorch自动求导与梯度路径](https://docs.pytorch.org/docs/stable/notes/autograd.html)。

**环境边界：**代码使用CPU float64，全部动作和状态可以局部核对；未实测GPU、自动混合精度、多进程rollout、跨设备复现或预训练语言模型。文献与开源库算法不等于本课已经运行过它们的完整训练系统。